# BugTwin

Semantic duplicate bug-report retrieval with a chronological evaluation harness.

**Environment:** Python 3 · Google Colab · one executable notebook.

**Start:** Run all cells. By default, BugTwin fetches the public Eclipse Platform bug corpus from the pinned BugHub repository revision. An offline fixture mode is available for development and CI (`BUGTWIN_M1_SOURCE=fixture`).

BugTwin currently includes canonical ingestion and auditing, chronological dataset splits, lexical retrieval (TF-IDF and BM25), benchmark reporting, regression tests, and an interactive search interface. A retrieved result is a candidate for human review, not a confirmed duplicate.

The benchmark includes only source-verified direct duplicate links and never treats unlabeled pairs as negatives. Raw downloaded data stays in the runtime directory.


**M3:** Pretrained MiniLM semantic retrieval, FAISS accelerated unrestricted search, RRF hybrid retrieval, optional cross-encoder reranking, chronological evaluation, and M3 regression tests. The CI fixture encoder is explicitly synthetic.


## Environment and dependencies

Dependencies are installed at runtime for standalone Colab execution. Versions are constrained to compatible major lines and recorded exactly at execution time.

In [ ]:
import os
import subprocess
import sys

# CI installs dependencies from requirements-ci.txt; Colab installs the inference stack here.
if os.getenv("BUGTWIN_SKIP_INSTALL", "0") != "1":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "pandas>=2.2,<3", "numpy>=1.26,<3", "requests>=2.31,<3",
        "matplotlib>=3.8,<4", "pytest>=8,<10", "scikit-learn>=1.4,<2",
        "scipy>=1.12,<2", "gradio>=5.0,<7.0",
        "sentence-transformers>=3.3,<6", "faiss-cpu>=1.9,<2",
    ])


In [ ]:
from __future__ import annotations

import csv
import gzip
import hashlib
import io
import json
import logging
import os
import platform
import re
import sys
import tarfile
import tempfile
import time
import zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any, Iterable, Mapping
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

LOGGER = logging.getLogger("bugtwin.m1")
if not LOGGER.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    LOGGER.addHandler(handler)
LOGGER.setLevel(logging.INFO)

@dataclass(frozen=True)
class M1Config:
    source_mode: str = os.getenv("BUGTWIN_M1_SOURCE", "bughub").strip().lower()
    local_root: str = os.getenv("BUGTWIN_M1_LOCAL_ROOT", "").strip()
    project: str = "EclipsePlatform"
    repository: str = "logpai/bughub"
    ref: str = "master"
    output_root: str = "/content/bugtwin_runtime" if Path("/content").is_dir() else str(Path.cwd() / "bugtwin_runtime")
    timeout_seconds: int = 45
    max_download_bytes: int = 350_000_000
    max_expanded_bytes: int = 750_000_000
    max_payloads: int = 100_000
    train_fraction: float = 0.70
    validation_fraction: float = 0.15
    demo_seed_rows: int = 3_000
    enforce_duplicate_labels: bool = True

CONFIG = M1Config()
if CONFIG.source_mode not in {"bughub", "local", "fixture"}:
    raise ValueError("BUGTWIN_M1_SOURCE must be bughub, local, or fixture")
ROOT = Path(CONFIG.output_root).expanduser().resolve()
RAW_DIR = ROOT / "data" / "raw"
NORMALIZED_DIR = ROOT / "data" / "normalized"
REPORT_DIR = ROOT / "reports" / "m1"
for folder in (RAW_DIR, NORMALIZED_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
LOGGER.info("M1 source=%s | output=%s", CONFIG.source_mode, ROOT)

## Source provenance and controlled acquisition

The source collector uses GitHub's repository tree API to discover actual artifacts. It pins the resolved commit SHA before downloading source files. The repository snapshot is logged, and each file receives a SHA-256 digest. Unsupported file formats stop the real-data gate instead of being interpreted speculatively.

In [ ]:
SUPPORTED_EXTENSIONS = (
    ".csv", ".tsv", ".json", ".jsonl", ".ndjson", ".xml",
    ".zip", ".tar", ".tar.gz", ".tgz", ".gz",
)

@dataclass(frozen=True)
class SourceArtifact:
    relative_path: str
    sha256: str
    byte_size: int
    origin: str


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def request_json(session: requests.Session, url: str, timeout: int) -> Any:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


def resolve_bughub_snapshot(
    session: requests.Session, config: M1Config,
) -> tuple[str, str, dict[str, Any]]:
    commit_url = f"https://api.github.com/repos/{config.repository}/commits/{config.ref}"
    commit_info = request_json(session, commit_url, config.timeout_seconds)
    commit_sha = str(commit_info.get("sha", ""))
    tree_sha = str(commit_info.get("commit", {}).get("tree", {}).get("sha", ""))
    if not re.fullmatch(r"[0-9a-f]{40}", commit_sha):
        raise RuntimeError("GitHub response did not provide a valid source commit SHA")
    if not re.fullmatch(r"[0-9a-f]{40}", tree_sha):
        raise RuntimeError("GitHub response did not provide a valid source tree SHA")

    tree_url = f"https://api.github.com/repos/{config.repository}/git/trees/{tree_sha}?recursive=1"
    tree = request_json(session, tree_url, config.timeout_seconds)
    if str(tree.get("sha", "")) != tree_sha:
        raise RuntimeError("Pinned source tree SHA mismatch; source integrity cannot be certified")
    if tree.get("truncated"):
        raise RuntimeError("Repository tree is truncated; source discovery cannot be certified")
    if not isinstance(tree.get("tree"), list):
        raise RuntimeError("GitHub response did not contain repository tree entries")
    return commit_sha, tree_sha, tree


def acquire_bughub(config: M1Config, destination: Path) -> tuple[list[SourceArtifact], dict[str, Any]]:
    session = requests.Session()
    session.headers.update({"Accept": "application/vnd.github+json", "User-Agent": "BugTwin-M1/1.0"})
    commit_sha, tree_sha, tree = resolve_bughub_snapshot(session, config)

    prefix = f"{config.project}/"
    entries = [entry for entry in tree["tree"]
               if entry.get("type") == "blob" and str(entry.get("path", "")).startswith(prefix)]
    eligible = [entry for entry in entries if str(entry["path"]).lower().endswith(SUPPORTED_EXTENSIONS)]
    if not eligible:
        names = [entry.get("path") for entry in entries[:25]]
        raise RuntimeError(f"No supported dataset artifacts beneath {prefix}: {names}")

    source_files: list[SourceArtifact] = []
    total_bytes = 0
    for entry in sorted(eligible, key=lambda item: str(item["path"])):
        relative = str(entry["path"])
        if "/" not in relative or ".." in PurePosixPath(relative).parts:
            raise RuntimeError(f"Unsafe repository path: {relative}")
        declared = int(entry.get("size", 0))
        total_bytes += declared
        if total_bytes > config.max_download_bytes:
            raise RuntimeError(f"Source files exceed max_download_bytes ({total_bytes:,} bytes)")
        local_path = destination / relative
        local_path.parent.mkdir(parents=True, exist_ok=True)
        uri = f"https://raw.githubusercontent.com/{config.repository}/{commit_sha}/{relative}"
        if local_path.exists() and local_path.stat().st_size == declared:
            digest = sha256_file(local_path)
        else:
            with session.get(uri, stream=True, timeout=config.timeout_seconds) as response:
                response.raise_for_status()
                chunks = 0
                with local_path.open("wb") as sink:
                    for part in response.iter_content(chunk_size=1024 * 1024):
                        if not part:
                            continue
                        chunks += len(part)
                        if chunks > max(declared + 1024, config.max_download_bytes):
                            raise RuntimeError(f"Unexpected download size: {relative}")
                        sink.write(part)
            digest = sha256_file(local_path)
        if declared and local_path.stat().st_size != declared:
            raise RuntimeError(f"Size mismatch downloading {relative}")
        source_files.append(SourceArtifact(relative, digest, local_path.stat().st_size, uri))
        LOGGER.info("Verified artifact: %s (%s bytes)", relative, local_path.stat().st_size)
    return source_files, {
        "repository": config.repository, "commit_sha": commit_sha, "tree_sha": tree_sha,
        "discovered_entries": len(entries), "recognized_artifacts": len(eligible),
        "source_kind": "github_pinned_commit",
    }


def acquire_local(path_string: str, destination: Path) -> tuple[list[SourceArtifact], dict[str, Any]]:
    root = Path(path_string).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"Local dataset path not found: {root}")
    candidates = [root] if root.is_file() else sorted(p for p in root.rglob("*") if p.is_file())
    candidates = [p for p in candidates if p.name.lower().endswith(SUPPORTED_EXTENSIONS)]
    if not candidates:
        raise ValueError("No supported data files found (CSV, TSV, JSON, XML, ZIP, TAR, GZ)")
    import shutil
    artifacts = []
    for file in candidates:
        name = file.name if root.is_file() else file.relative_to(root).as_posix()
        target = destination / "local" / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if file.resolve() != target.resolve():
            shutil.copyfile(file, target)
        artifacts.append(SourceArtifact(f"local/{name}", sha256_file(target), target.stat().st_size, str(file)))
    return artifacts, {"source_kind": "local", "origin": str(root)}


def fixture_csv(destination: Path) -> tuple[list[SourceArtifact], dict[str, Any]]:
    rows = [
        {"bug_id": "101", "short_desc": "Login screen becomes unresponsive", "description": "UI freezes after password recovery", "creation_time": "2021-01-01T10:00:00Z", "duplicate_of": "", "component": "Authentication"},
        {"bug_id": "102", "short_desc": "Export button does not work", "description": "CSV export raises an error", "creation_time": "2021-02-01T10:00:00Z", "duplicate_of": "", "component": "Export"},
        {"bug_id": "103", "short_desc": "Sign-in hangs after a password reset", "description": "Login UI remains frozen", "creation_time": "2021-03-01T10:00:00Z", "duplicate_of": "101", "component": "Authentication"},
        {"bug_id": "104", "short_desc": "Export fails for large documents", "description": "An error appears in CSV export", "creation_time": "2021-04-01T10:00:00Z", "duplicate_of": "102", "component": "Export"},
        {"bug_id": "105", "short_desc": "Settings page is slow", "description": "Delayed response", "creation_time": "2021-05-01T10:00:00Z", "duplicate_of": "", "component": "Settings"},
        {"bug_id": "106", "short_desc": "Password reset locks login view", "description": "Unable to type credentials", "creation_time": "2021-06-01T10:00:00Z", "duplicate_of": "103", "component": "Authentication"},
        {"bug_id": "107", "short_desc": "Preferences dialog displays blank", "description": "Empty settings form", "creation_time": "2021-07-01T10:00:00Z", "duplicate_of": "", "component": "Settings"},
        {"bug_id": "108", "short_desc": "CSV export crashes", "description": "Unable to save export", "creation_time": "2021-08-01T10:00:00Z", "duplicate_of": "102", "component": "Export"},
        {"bug_id": "109", "short_desc": "Opening logs throws exception", "description": "NullPointerException in LogView.java", "creation_time": "2021-09-01T10:00:00Z", "duplicate_of": "", "component": "Logs"},
        {"bug_id": "110", "short_desc": "Preferences blank after restart", "description": "Empty settings view", "creation_time": "2021-10-01T10:00:00Z", "duplicate_of": "107", "component": "Settings"},
    ]
    path = destination / "fixture" / "verified_small_fixture.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)
    return [SourceArtifact("fixture/verified_small_fixture.csv", sha256_file(path), path.stat().st_size, "internal_test_fixture")], {"source_kind": "synthetic_fixture", "is_real_dataset": False}

## Safe file readers and schema discovery

Source files are treated as untrusted inputs. This loader reads CSV, JSON, JSON Lines, Bugzilla-style XML, and common archives; it does not execute serialized Python objects. Known column aliases are mapped only when unambiguous. The resulting mapping and per-artifact row counts are part of the audit.
The Eclipse Platform archive contains issue records. Its `train.csv` and `test.csv` files contain
`Issue_id`/`Duplicate` labels only and are audited separately rather than parsed as bug reports.
These publisher labels are not duplicate-target links; historical retrieval continues to use
verified earlier target identifiers.


In [ ]:
@dataclass(frozen=True)
class Payload:
    name: str
    data: bytes


def validate_archive_member(name: str) -> bool:
    path = PurePosixPath(name.replace("\\", "/"))
    return not path.is_absolute() and ".." not in path.parts and bool(path.name)


def expand_payload(name: str, data: bytes, *, limit: int, max_files: int) -> list[Payload]:
    expanded: list[Payload] = []
    extracted_bytes = 0

    def append(member: str, content: bytes) -> None:
        nonlocal extracted_bytes
        if not validate_archive_member(member):
            raise ValueError(f"Unsafe archive member: {member}")
        extracted_bytes += len(content)
        if extracted_bytes > limit:
            raise ValueError("Archive expansion exceeds configured byte limit")
        if len(expanded) >= max_files:
            raise ValueError("Archive contains more files than configured safety limit")
        if member.lower().endswith(SUPPORTED_EXTENSIONS):
            expanded.extend(expand_payload(member, content, limit=limit-extracted_bytes+len(content), max_files=max_files-len(expanded)))

    lower = name.lower()
    if lower.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            for member in archive.infolist():
                if member.is_dir():
                    continue
                if member.file_size > limit:
                    raise ValueError("Archive member is larger than the allowed expanded size")
                with archive.open(member) as stream:
                    contents = stream.read(limit + 1)
                if len(contents) > limit:
                    raise ValueError("Archive member decompression exceeds maximum")
                append(f"{name}/{member.filename}", contents)
    elif lower.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(fileobj=io.BytesIO(data), mode="r:*") as archive:
            for member in archive.getmembers():
                if not member.isfile():
                    continue
                if member.size > limit:
                    raise ValueError("Tar member is too large")
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                contents = stream.read(limit + 1)
                append(f"{name}/{member.name}", contents)
    elif lower.endswith(".gz"):
        with gzip.GzipFile(fileobj=io.BytesIO(data)) as stream:
            contents = stream.read(limit + 1)
        if len(contents) > limit:
            raise ValueError("Gzip decompression exceeds configured size")
        expanded.extend(expand_payload(name[:-3], contents, limit=limit, max_files=max_files))
    else:
        expanded.append(Payload(name, data))
    return expanded


def read_text_bytes(data: bytes) -> str:
    try:
        return data.decode("utf-8-sig")
    except UnicodeDecodeError:
        LOGGER.warning("Input is not UTF-8; falling back to Latin-1")
        return data.decode("latin-1")


def local_name(tag: str) -> str:
    return tag.rsplit("}", 1)[-1].lower()


def read_xml_reports(data: bytes) -> pd.DataFrame:
    root = ET.fromstring(data)
    entries = [node for node in root.iter() if local_name(node.tag) in {"bug", "bugreport", "issue", "report"}]
    if not entries and local_name(root.tag) in {"bug", "bugreport", "issue", "report"}:
        entries = [root]
    records = []
    for item in entries:
        record: dict[str, str] = {}
        for child in item:
            key = local_name(child.tag)
            if key in {"long_desc", "comments"}:
                continue  # Comment history can postdate report creation.
            if len(child) == 0:
                record[key] = (child.text or "").strip()
            elif key in {"description", "short_desc"}:
                record[key] = " ".join(segment.strip() for segment in child.itertext() if segment.strip())
        if record:
            records.append(record)
    if not records:
        raise ValueError("XML does not contain recognized bug/issue/report records")
    return pd.DataFrame(records)


def read_table(payload: Payload) -> pd.DataFrame:
    name = payload.name.lower()
    if name.endswith((".csv", ".tsv")):
        text = read_text_bytes(payload.data)
        delimiter = "\t" if name.endswith(".tsv") else ","
        if name.endswith(".csv"):
            try:
                delimiter = csv.Sniffer().sniff(text[:32_768], delimiters=",;\t|").delimiter
            except csv.Error:
                pass
        return pd.read_csv(io.StringIO(text), sep=delimiter, dtype=str, keep_default_na=False, on_bad_lines="error")
    if name.endswith((".jsonl", ".ndjson")):
        return pd.read_json(io.StringIO(read_text_bytes(payload.data)), lines=True, dtype=False).astype(object)
    if name.endswith(".json"):
        document = json.loads(read_text_bytes(payload.data))
        if isinstance(document, dict):
            for field in ("bugs", "issues", "reports", "records", "data", "items"):
                if isinstance(document.get(field), list):
                    document = document[field]
                    break
            else:
                document = [document]
        if not isinstance(document, list):
            raise ValueError("JSON content must contain a list of bug-report records")
        return pd.DataFrame(document)
    if name.endswith(".xml"):
        return read_xml_reports(payload.data)
    raise ValueError(f"Unsupported payload type: {payload.name}")


FIELD_ALIASES = {
    "report_id": ("report_id", "bug_id", "bugid", "issue_id", "issueid", "id", "bug_number", "issue_number"),
    "title": ("title", "summary", "short_desc", "short_description", "shortdescription", "bug_title", "subject"),
    "description": ("description", "long_description", "long_desc", "bug_description", "body", "details", "first_comment"),
    "created_at": ("created_at", "creation_time", "creation_date", "created", "opened_at", "open_date", "date_opened", "creation_ts", "date_created", "timestamp", "creationdate", "created_time"),
    "duplicate_of": ("duplicate_of", "duplicate_id", "duplicateid", "duplicate_bug_id", "dup_id", "dup_id1", "duplicated_issue", "master_id", "dup_bug_id", "duplicate_target"),
    "component": ("component", "component_name", "subcategory"),
    "product": ("product", "product_name"),
    "severity": ("severity", "bug_severity", "priority"),
    "source_url": ("source_url", "bug_url", "url", "issue_url"),
}


def normalized_column(name: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def infer_field_map(columns: Iterable[str]) -> dict[str, str]:
    names: dict[str, list[str]] = {}
    for column in columns:
        names.setdefault(normalized_column(column), []).append(str(column))
    result: dict[str, str] = {}
    for field, aliases in FIELD_ALIASES.items():
        candidates = [names[a][0] for a in aliases if a in names and len(names[a]) == 1]
        if len(candidates) > 1:
            # Explicit canonical field takes precedence; otherwise ambiguous fields require correction.
            canonical = names.get(field, [])
            if len(canonical) == 1:
                result[field] = canonical[0]
                continue
            raise ValueError(f"Ambiguous mapping for {field}: {candidates}")
        if candidates:
            result[field] = candidates[0]
    required = {"report_id", "title", "created_at"}
    missing = required - result.keys()
    if missing:
        raise ValueError(f"Missing required source fields {sorted(missing)}; available columns: {list(columns)}")
    return result


def parse_timestamp(value: Any) -> pd.Timestamp:
    string = str(value).strip()
    if not string:
        return pd.NaT
    if re.fullmatch(r"\d{10}(?:\d{3})?", string):
        unit = "ms" if len(string) == 13 else "s"
        return pd.to_datetime(int(string), unit=unit, utc=True, errors="coerce")
    return pd.to_datetime(string, utc=True, errors="coerce", format="mixed")


def clean_id(value: Any) -> str:
    result = str(value).strip()
    if re.fullmatch(r"\d+\.0", result):
        return result[:-2]
    return result


def optional_str(value: Any) -> str:
    if value is None or (not isinstance(value, (str, dict, list)) and pd.isna(value)):
        return ""
    return str(value).strip()


def parse_duplicate_targets(value: Any) -> list[str]:
    raw = optional_str(value)
    if not raw or raw.lower() in {"null", "none", "nan", "n/a", "0", "-1", "false", "no", "[]"}:
        return []
    if raw.startswith("[") and raw.endswith("]"):
        try:
            loaded = json.loads(raw)
            if isinstance(loaded, list):
                return [clean_id(item) for item in loaded if clean_id(item)]
        except json.JSONDecodeError:
            pass
    targets = [clean_id(part) for part in re.split(r"[|;,]", raw) if clean_id(part)]
    return list(dict.fromkeys(targets))


def adapt_report_table(raw: pd.DataFrame, project: str, source_file: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    mapping = infer_field_map(raw.columns)
    transformed = pd.DataFrame(index=raw.index)
    for field in FIELD_ALIASES:
        transformed[field] = raw[mapping[field]] if field in mapping else ""
    transformed["project"] = project
    transformed["report_id"] = transformed["report_id"].map(clean_id)
    transformed["title"] = transformed["title"].map(optional_str)
    transformed["description"] = transformed["description"].map(optional_str)
    transformed["created_at"] = transformed["created_at"].map(parse_timestamp)
    for field in ("component", "product", "severity", "source_url", "duplicate_of"):
        transformed[field] = transformed[field].map(optional_str)
    transformed["record_key"] = transformed["project"] + "::" + transformed["report_id"]
    transformed["source_file"] = source_file
    transformed["source_row"] = raw.index.astype(str)
    transformed["search_text"] = (transformed["title"] + "\n" + transformed["description"]).str.strip()
    problems = pd.Series("", index=transformed.index, dtype="object")
    problems.loc[transformed["report_id"].isin(["", "nan", "None"])] += "missing_id;"
    problems.loc[transformed["title"].eq("") & transformed["description"].eq("")] += "missing_text;"
    problems.loc[transformed["created_at"].isna()] += "invalid_created_at;"
    quarantined = transformed.loc[problems.ne("")].copy()
    quarantined["reason"] = problems.loc[problems.ne("")]
    clean = transformed.loc[problems.eq("")].copy().reset_index(drop=True)
    audit = {
        "source_file": source_file, "raw_rows": int(len(raw)),
        "valid_rows": int(len(clean)), "quarantined_rows": int(len(quarantined)),
        "field_mapping": mapping, "input_columns": list(map(str, raw.columns)),
    }
    return clean, quarantined, audit

## Canonical reports and duplicate-link audit

Direct source links define the verified positive evidence; transitive links are **not** automatically inferred. An edge pointing to a newer report is kept for audit but is not an eligible historical target. Missing links are never rewritten as negative labels.

In [ ]:
REPORT_COLUMNS = [
    "record_key", "project", "report_id", "title", "description", "created_at",
    "component", "product", "severity", "source_url", "duplicate_of",
    "source_file", "source_row", "search_text",
]


def consolidate_reports(frames: list[pd.DataFrame]) -> tuple[pd.DataFrame, dict[str, int]]:
    if not frames:
        raise ValueError("No successfully adapted bug-report tables")
    reports = pd.concat(frames, ignore_index=True)
    if reports.empty:
        raise ValueError("All loaded reports were quarantined")
    # Identical repeated rows may occur in source train/test file exports.
    duplicates = reports.duplicated(subset=["record_key"], keep=False)
    repeated = reports.loc[duplicates]
    collapsed = 0
    if not repeated.empty:
        payload_fields = ["project", "report_id", "title", "description", "created_at", "duplicate_of"]
        for key, group in repeated.groupby("record_key", sort=False):
            if group[payload_fields].drop_duplicates().shape[0] > 1:
                raise ValueError(f"Conflicting source records for report {key}; manual source reconciliation required")
        before = len(reports)
        reports = reports.drop_duplicates(subset=["record_key"], keep="first")
        collapsed = before - len(reports)
    reports = reports.sort_values(["created_at", "record_key"], kind="mergesort").reset_index(drop=True)
    if reports["record_key"].duplicated().any():
        raise AssertionError("Canonical keys are not unique")
    return reports[REPORT_COLUMNS].copy(), {"collapsed_identical_records": int(collapsed), "canonical_rows": int(len(reports))}


def audit_duplicate_edges(reports: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    by_key = reports.set_index("record_key")
    edge_records = []
    for row in reports.itertuples(index=False):
        for target_id in parse_duplicate_targets(row.duplicate_of):
            target_key = target_id if "::" in target_id else f"{row.project}::{target_id}"
            if target_key == row.record_key:
                status = "self_reference"
            elif target_key not in by_key.index:
                status = "missing_target"
            else:
                target_time = by_key.at[target_key, "created_at"]
                if target_time < row.created_at:
                    status = "eligible_direct_positive"
                elif target_time == row.created_at:
                    status = "timestamp_tie"
                else:
                    status = "future_target"
            edge_records.append({
                "source_key": row.record_key, "target_key": target_key,
                "source_created_at": row.created_at, "status": status,
                "source_split": "",
            })
    columns = ["source_key", "target_key", "source_created_at", "status", "source_split"]
    edges = pd.DataFrame(edge_records, columns=columns)
    direct = edges.loc[edges["status"].eq("eligible_direct_positive")].copy()
    return edges, direct


def chronological_splits(reports: pd.DataFrame, train_fraction: float, validation_fraction: float) -> pd.DataFrame:
    if not (0 < train_fraction < 1 and 0 < validation_fraction < 1 and train_fraction + validation_fraction < 1):
        raise ValueError("Invalid chronological split fractions")
    frame = reports[["record_key", "created_at"]].sort_values(["created_at", "record_key"], kind="mergesort").copy()
    if len(frame) < 6:
        raise ValueError("At least six valid reports are required for a three-way split")
    # Assign equal-day reports to the same partition to eliminate boundary ties.
    day = frame["created_at"].dt.floor("D")
    days = pd.DataFrame({"day": day}).value_counts().sort_index()
    if len(days) < 3:
        raise ValueError("A three-way chronological split requires at least three distinct report dates")
    sizes = days.to_numpy()
    cumulative = np.cumsum(sizes)
    n = len(frame)
    first_boundary = int(np.argmin(np.abs(cumulative[:-2] - train_fraction * n))) + 1
    second_candidates = np.arange(first_boundary + 1, len(days))
    second_boundary = int(second_candidates[np.argmin(np.abs(cumulative[second_candidates - 1] - (train_fraction + validation_fraction) * n))])
    ordered_days = days.index.get_level_values("day")
    train_last = ordered_days[first_boundary - 1]
    validation_last = ordered_days[second_boundary - 1]
    frame["split"] = np.where(day <= train_last, "train", np.where(day <= validation_last, "validation", "test"))
    if set(frame["split"]) != {"train", "validation", "test"}:
        raise AssertionError("Chronological partition is empty")
    return frame[["record_key", "split"]].reset_index(drop=True)


def build_query_registry(reports: pd.DataFrame, edges: pd.DataFrame, split_table: pd.DataFrame) -> pd.DataFrame:
    positives = edges.loc[edges["status"].eq("eligible_direct_positive")].groupby("source_key")["target_key"].agg(lambda seq: sorted(set(seq)))
    query_table = reports[["record_key", "project", "report_id", "created_at"]].merge(
        split_table, on="record_key", how="left", validate="one_to_one"
    )
    query_table["positive_target_keys"] = query_table["record_key"].map(positives)
    query_table["positive_target_keys"] = query_table["positive_target_keys"].map(lambda item: item if isinstance(item, list) else [])
    query_table["positive_target_count"] = query_table["positive_target_keys"].map(len)
    query_table["ground_truth_status"] = np.where(
        query_table["positive_target_count"] > 0, "eligible_positive", "unknown_or_no_eligible_positive"
    )
    return query_table


def historical_candidates(reports: pd.DataFrame, query_key: str, project_restriction: bool = True) -> pd.DataFrame:
    row = reports.loc[reports["record_key"].eq(query_key)]
    if len(row) != 1:
        raise KeyError(f"Query key not in corpus: {query_key}")
    query = row.iloc[0]
    mask = reports["created_at"].lt(query["created_at"]) & reports["record_key"].ne(query_key)
    if project_restriction:
        mask &= reports["project"].eq(query["project"])
    return reports.loc[mask].copy()


def sample_link_preserving_cohort(reports: pd.DataFrame, edges: pd.DataFrame, seed_size: int) -> pd.DataFrame:
    if seed_size < 1:
        raise ValueError("seed_size must be positive")
    ordered = reports.sort_values(["created_at", "record_key"], kind="mergesort")
    seeds = set(ordered.tail(seed_size)["record_key"])
    links = edges.loc[edges["status"].eq("eligible_direct_positive")]
    necessary = set(links.loc[links["source_key"].isin(seeds), "target_key"])
    return ordered.loc[ordered["record_key"].isin(seeds | necessary)].copy()

## Data engineering regression tests

Tests below are independent of the remote dataset. They validate schema mapping, timestamp ordering, link direction, duplicate integrity, historical eligibility, deterministic splits, and file-reader safeguards. Fixture passes **do not** certify BugHub's real source schema.

In [ ]:
def test_schema_mapping() -> None:
    mapping = infer_field_map(["bug_id", "short_desc", "creation_time", "dup_id"])
    assert mapping["report_id"] == "bug_id"
    assert mapping["title"] == "short_desc"
    assert mapping["duplicate_of"] == "dup_id"


def test_invalid_schema() -> None:
    try:
        infer_field_map(["bug_id", "summary"])
    except ValueError as error:
        assert "created_at" in str(error)
    else:
        raise AssertionError("Invalid schema was accepted")


def test_duplicate_parsing() -> None:
    assert parse_duplicate_targets("[12, 13]") == ["12", "13"]
    assert parse_duplicate_targets("12|13|12") == ["12", "13"]
    assert parse_duplicate_targets("-1") == []


def test_timestamp_conversion() -> None:
    assert parse_timestamp("2022-01-01").tzinfo is not None
    assert parse_timestamp("1640995200") == pd.Timestamp("2022-01-01T00:00:00Z")
    assert pd.isna(parse_timestamp("not-a-date"))


def test_canonical_and_edges() -> None:
    with tempfile.TemporaryDirectory() as root:
        artifacts, _ = fixture_csv(Path(root))
        path = Path(root) / artifacts[0].relative_path
        frame, quarantine, _ = adapt_report_table(pd.read_csv(path, dtype=str, keep_default_na=False), "EclipsePlatform", path.name)
        assert quarantine.empty
        reports, counts = consolidate_reports([frame])
        assert counts["canonical_rows"] == 10
        edges, direct = audit_duplicate_edges(reports)
        assert len(direct) == 5
        assert edges["status"].eq("eligible_direct_positive").all()
        splits = chronological_splits(reports, 0.7, 0.15)
        assert set(splits["split"]) == {"train", "validation", "test"}
        q = build_query_registry(reports, edges, splits)
        assert q.loc[q["report_id"].eq("108"), "positive_target_count"].iloc[0] == 1
        eligible = historical_candidates(reports, "EclipsePlatform::108")
        assert "EclipsePlatform::102" in set(eligible["record_key"])
        assert "EclipsePlatform::109" not in set(eligible["record_key"])
        cohort = sample_link_preserving_cohort(reports, edges, 2)
        assert "EclipsePlatform::107" in set(cohort["record_key"])


def test_bad_row_quarantine() -> None:
    input_table = pd.DataFrame({"bug_id": ["a", "b"], "summary": ["Missing date", "Valid record"],
                                "creation_time": ["not-a-date", "2020-02-01"]})
    valid, quarantine, _ = adapt_report_table(input_table, "P", "fixture.csv")
    assert len(valid) == 1 and len(quarantine) == 1
    assert quarantine.iloc[0]["reason"] == "invalid_created_at;"


def test_conflicting_identifiers_rejected() -> None:
    raw = pd.DataFrame({"report_id": ["1", "1"], "title": ["A", "B"],
                        "created_at": ["2020-01-01", "2020-01-01"]})
    clean, _, _ = adapt_report_table(raw, "P", "conflict.csv")
    try:
        consolidate_reports([clean])
    except ValueError as error:
        assert "Conflicting" in str(error)
    else:
        raise AssertionError("Conflicting IDs were not rejected")


def test_future_and_missing_edges() -> None:
    raw = pd.DataFrame({"report_id": ["1", "2", "3", "4"],
                        "title": ["A", "B", "C", "D"],
                        "created_at": ["2020-01-01", "2020-02-01", "2020-03-01", "2020-04-01"],
                        "duplicate_of": ["3", "99", "3", "1"]})
    clean, _, _ = adapt_report_table(raw, "P", "edges.csv")
    reports, _ = consolidate_reports([clean])
    edges, _ = audit_duplicate_edges(reports)
    assert set(edges["status"]) == {"future_target", "missing_target", "self_reference", "eligible_direct_positive"}


def test_archive_path_safety() -> None:
    assert validate_archive_member("safe/report.csv")
    assert not validate_archive_member("../outside.csv")
    with io.BytesIO() as stream:
        with zipfile.ZipFile(stream, "w") as archive:
            archive.writestr("../../escape.csv", "x,y\n1,2")
        try:
            expand_payload("unsafe.zip", stream.getvalue(), limit=10000, max_files=10)
        except ValueError as error:
            assert "Unsafe" in str(error)
        else:
            raise AssertionError("Traversal payload was accepted")


def test_tie_break_and_strict_time() -> None:
    raw = pd.DataFrame({"bug_id": ["1", "2", "3", "4", "5", "6"],
                        "summary": list("ABCDEF"),
                        "creation_time": ["2020-01-01", "2020-01-01", "2020-02-01", "2020-03-01", "2020-04-01", "2020-05-01"]})
    reports, _, _ = adapt_report_table(raw, "P", "ties.csv")
    reports, _ = consolidate_reports([reports])
    assert "P::1" not in set(historical_candidates(reports, "P::2")["record_key"])
    result = chronological_splits(reports, .5, .25)
    assert result.loc[result["record_key"].eq("P::1"), "split"].iloc[0] == result.loc[result["record_key"].eq("P::2"), "split"].iloc[0]



def test_table_readers_and_archives() -> None:
    csv_blob = b"bug_id,summary,creation_time\n7,Memory issue,2020-01-02\n"
    rows = read_table(Payload("records.csv", csv_blob))
    assert rows.loc[0, "summary"] == "Memory issue"
    json_blob = json.dumps({"reports": [{"bug_id": "7", "summary": "Memory issue"}]}).encode()
    json_rows = read_table(Payload("records.json", json_blob))
    assert json_rows.loc[0, "bug_id"] == "7"
    with io.BytesIO() as buffer:
        with zipfile.ZipFile(buffer, "w") as archive:
            archive.writestr("data/records.csv", csv_blob)
        expanded = expand_payload("archive.zip", buffer.getvalue(), limit=10_000, max_files=5)
    assert len(expanded) == 1
    assert read_table(expanded[0]).loc[0, "bug_id"] == "7"


def test_direct_evidence_no_transitive_inference() -> None:
    with tempfile.TemporaryDirectory() as root:
        artifacts, _ = fixture_csv(Path(root))
        path = Path(root) / artifacts[0].relative_path
        raw = pd.read_csv(path, dtype=str, keep_default_na=False)
        normalized, _, _ = adapt_report_table(raw, "EclipsePlatform", "fixture.csv")
        reports, _ = consolidate_reports([normalized])
        edges, _ = audit_duplicate_edges(reports)
        splits = chronological_splits(reports, .7, .15)
        registry = build_query_registry(reports, edges, splits)
        targets = registry.loc[registry["record_key"].eq("EclipsePlatform::106"), "positive_target_keys"].iloc[0]
        assert targets == ["EclipsePlatform::103"]
        assert "EclipsePlatform::101" not in targets


def test_unlabeled_reports_remain_unknown() -> None:
    raw = pd.DataFrame({
        "report_id": ["1", "2", "3", "4", "5", "6"],
        "title": list("ABCDEF"),
        "created_at": [f"2020-01-{i:02d}" for i in range(1, 7)],
    })
    clean, _, _ = adapt_report_table(raw, "P", "unlabeled.csv")
    reports, _ = consolidate_reports([clean])
    edges, _ = audit_duplicate_edges(reports)
    splits = chronological_splits(reports, .5, .25)
    queries = build_query_registry(reports, edges, splits)
    assert queries["ground_truth_status"].eq("unknown_or_no_eligible_positive").all()


def test_tied_target_is_ineligible() -> None:
    raw = pd.DataFrame({
        "report_id": ["1", "2"], "title": ["A", "B"],
        "created_at": ["2020-01-01T12:00:00Z"] * 2,
        "duplicate_of": ["", "1"],
    })
    clean, _, _ = adapt_report_table(raw, "P", "same_time.csv")
    reports, _ = consolidate_reports([clean])
    edges, direct = audit_duplicate_edges(reports)
    assert len(direct) == 0 and edges.iloc[0]["status"] == "timestamp_tie"


def test_invalid_split_parameters() -> None:
    with tempfile.TemporaryDirectory() as root:
        artifacts, _ = fixture_csv(Path(root))
        raw = pd.read_csv(Path(root) / artifacts[0].relative_path, dtype=str, keep_default_na=False)
        clean, _, _ = adapt_report_table(raw, "P", "fixture.csv")
        reports, _ = consolidate_reports([clean])
        try:
            chronological_splits(reports, .9, .2)
        except ValueError as error:
            assert "Invalid" in str(error)
        else:
            raise AssertionError("Overlapping partition fractions were accepted")


def test_source_resolves_pinned_commit_tree() -> None:
    from unittest.mock import patch

    commit_sha = "a" * 40
    tree_sha = "b" * 40
    requests_seen = []

    def fake_request(_session: Any, url: str, _timeout: int) -> dict[str, Any]:
        requests_seen.append(url)
        if url.endswith("/commits/master"):
            return {"sha": commit_sha, "commit": {"tree": {"sha": tree_sha}}}
        if url.endswith(f"/git/trees/{tree_sha}?recursive=1"):
            return {"sha": tree_sha, "truncated": False, "tree": []}
        raise AssertionError(f"Unexpected GitHub request: {url}")

    with patch("__main__.request_json", side_effect=fake_request):
        result = resolve_bughub_snapshot(requests.Session(), M1Config())
    assert result[0] == commit_sha and result[1] == tree_sha
    assert len(requests_seen) == 2
    assert f"/git/trees/{tree_sha}?recursive=1" in requests_seen[1]


def test_source_rejects_mismatched_pinned_tree() -> None:
    from unittest.mock import patch

    tree_sha = "b" * 40
    responses = [
        {"sha": "a" * 40, "commit": {"tree": {"sha": tree_sha}}},
        {"sha": "c" * 40, "truncated": False, "tree": []},
    ]
    with patch("__main__.request_json", side_effect=responses):
        try:
            resolve_bughub_snapshot(requests.Session(), M1Config())
        except RuntimeError as error:
            assert "tree SHA mismatch" in str(error)
        else:
            raise AssertionError("Mismatched source tree was accepted")


def test_bughub_issue_fields_and_auxiliary_splits() -> None:
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        main = pd.DataFrame({
            "Issue_id": ["101", "102", "103", "104", "105", "106"],
            "Title": ["Login error", "Export freezes", "Login hangs", "Open view", "Export failure", "Settings blank"],
            "Description": ["Password reset", "CSV issue", "Password reset", "UI view", "CSV issue", "Preferences"],
            "Created_time": [f"2021-0{i}-10T10:00:00Z" for i in range(1, 7)],
            "Duplicated_issue": ["", "", "101", "", "102", ""],
            "Component": ["UI"] * 6,
        })
        archive_path = root / "EclipsePlatform" / "eclipse_platform.zip"
        archive_path.parent.mkdir(parents=True)
        with zipfile.ZipFile(archive_path, "w") as archive:
            archive.writestr("eclipse_platform.csv", main.to_csv(index=False))
        (root / "EclipsePlatform" / "train.csv").write_text(
            "Issue_id,Duplicate\n101,0\n102,0\n103,1\n104,0\n", encoding="utf-8"
        )
        (root / "EclipsePlatform" / "test.csv").write_text(
            "Issue_id,Duplicate\n105,1\n106,0\n", encoding="utf-8"
        )
        artifacts = [SourceArtifact(
            file.relative_to(root).as_posix(), sha256_file(file), file.stat().st_size, str(file)
        ) for file in sorted((root / "EclipsePlatform").iterdir())]
        reports, quarantine, audits, labels = load_and_adapt(artifacts, M1Config(), source_root=root)
        assert len(reports) == 6 and quarantine.empty
        assert reports.loc[reports["report_id"].eq("103"), "duplicate_of"].iloc[0] == "101"
        assert len(labels) == 6 and labels["matches_canonical_record"].all()
        assert labels["official_split"].value_counts().to_dict() == {"train": 4, "test": 2}
        assert labels["duplicate_flag"].sum() == 2
        assert len([entry for entry in audits if entry.get("role") == "official_split_labels"]) == 2
        edges, direct = audit_duplicate_edges(reports)
        assert len(edges) == 2 and len(direct) == 2


def test_bughub_split_labels_reject_ambiguous_values() -> None:
    bad = pd.DataFrame({"Issue_id": ["101"], "Duplicate": ["maybe"]})
    assert is_official_split_label_table("EclipsePlatform/train.csv", bad)
    try:
        adapt_official_split_labels(bad, "EclipsePlatform", "EclipsePlatform/train.csv")
    except ValueError as error:
        assert "Unrecognized duplicate flags" in str(error)
    else:
        raise AssertionError("Unrecognized labels were accepted")


def test_bughub_split_files_are_not_report_rows() -> None:
    label_data = pd.DataFrame({"Issue_id": ["101"], "Duplicate": ["1"]})
    assert is_official_split_label_table("EclipsePlatform/test.csv", label_data)
    assert not is_official_split_label_table("EclipsePlatform/bugs.csv", label_data)
    assert not is_official_split_label_table("EclipsePlatform/train.csv", pd.DataFrame({
        "Issue_id": ["101"], "Title": ["Bug"], "Created_time": ["2020-01-01"],
    }))


TESTS = [
    test_schema_mapping, test_invalid_schema, test_duplicate_parsing,
    test_timestamp_conversion, test_canonical_and_edges, test_bad_row_quarantine,
    test_conflicting_identifiers_rejected, test_future_and_missing_edges,
    test_archive_path_safety, test_tie_break_and_strict_time,
    test_table_readers_and_archives, test_direct_evidence_no_transitive_inference,
    test_unlabeled_reports_remain_unknown, test_tied_target_is_ineligible,
    test_invalid_split_parameters,
    test_source_resolves_pinned_commit_tree,
    test_source_rejects_mismatched_pinned_tree,
    test_bughub_issue_fields_and_auxiliary_splits,
    test_bughub_split_labels_reject_ambiguous_values,
    test_bughub_split_files_are_not_report_rows,
]

def run_m1_tests() -> pd.DataFrame:
    rows = []
    for test in TESTS:
        try:
            test()
            rows.append({"test": test.__name__, "status": "PASS", "details": ""})
        except Exception as error:
            rows.append({"test": test.__name__, "status": "FAIL", "details": f"{type(error).__name__}: {error}"})
    outcome = pd.DataFrame(rows)
    display(outcome)
    if not outcome["status"].eq("PASS").all():
        raise AssertionError("One or more M1 regression tests failed")
    return outcome


## Data ingestion and canonicalization

The acquisition gate is strict: failure stops the run and shows the source of the problem. If BugHub's actual files use an unmapped schema, inspect the printed source report and implement a documented adapter; do not coerce unrelated columns to make the test pass.

In [ ]:
def acquire_source(config: M1Config) -> tuple[list[SourceArtifact], dict[str, Any]]:
    if config.source_mode == "bughub":
        return acquire_bughub(config, RAW_DIR)
    if config.source_mode == "local":
        if not config.local_root:
            raise ValueError("BUGTWIN_M1_LOCAL_ROOT is required when BUGTWIN_M1_SOURCE=local")
        return acquire_local(config.local_root, RAW_DIR)
    return fixture_csv(RAW_DIR)


def is_official_split_label_table(source_file: str, raw_table: pd.DataFrame) -> bool:
    name = PurePosixPath(source_file).name.casefold()
    columns = {normalized_column(column) for column in raw_table.columns}
    return name in {"train.csv", "test.csv"} and columns == {"issue_id", "duplicate"}


def adapt_official_split_labels(raw_table: pd.DataFrame, project: str, source_file: str) -> pd.DataFrame:
    if not is_official_split_label_table(source_file, raw_table):
        raise ValueError(f"Not an official issue-label table: {source_file}")
    columns = {normalized_column(name): name for name in raw_table.columns}
    identifiers = raw_table[columns["issue_id"]].map(clean_id)
    flags = raw_table[columns["duplicate"]].astype(str).str.strip().str.casefold()
    unknown = ~flags.isin({"0", "1", "false", "true"})
    if unknown.any():
        raise ValueError(f"Unrecognized duplicate flags in {source_file}: {sorted(flags[unknown].unique())[:5]}")
    if identifiers.isin({"", "nan", "None"}).any():
        raise ValueError(f"Missing issue identifiers in {source_file}")
    if identifiers.duplicated().any():
        raise ValueError(f"Repeated issue identifiers in {source_file}")
    split = PurePosixPath(source_file).stem.casefold()
    return pd.DataFrame({
        "record_key": project + "::" + identifiers.to_numpy(),
        "official_split": split,
        "duplicate_flag": flags.isin({"1", "true"}).astype(np.int8).to_numpy(),
        "source_file": source_file,
    })


def load_and_adapt(
    artifacts: list[SourceArtifact],
    config: M1Config,
    source_root: Path | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, list[dict[str, Any]], pd.DataFrame]:
    root = RAW_DIR if source_root is None else source_root
    canonical_frames: list[pd.DataFrame] = []
    quarantined_frames: list[pd.DataFrame] = []
    official_label_frames: list[pd.DataFrame] = []
    file_audits: list[dict[str, Any]] = []
    payload_count = 0
    errors: list[str] = []
    for artifact in artifacts:
        path = root / artifact.relative_path
        raw = path.read_bytes()
        if sha256_bytes(raw) != artifact.sha256:
            raise RuntimeError(f"Checksum mismatch: {artifact.relative_path}")
        payloads = expand_payload(
            artifact.relative_path, raw,
            limit=config.max_expanded_bytes, max_files=config.max_payloads,
        )
        payload_count += len(payloads)
        if payload_count > config.max_payloads:
            raise RuntimeError("Exceeded total payload limit")
        for payload in payloads:
            try:
                raw_table = read_table(payload)
                if raw_table.empty:
                    raise ValueError("Table has no records")
                if is_official_split_label_table(payload.name, raw_table):
                    labels = adapt_official_split_labels(raw_table, config.project, payload.name)
                    official_label_frames.append(labels)
                    file_audits.append({
                        "source_file": payload.name,
                        "role": "official_split_labels",
                        "split": labels["official_split"].iloc[0],
                        "rows": len(labels),
                        "positive_flags": int(labels["duplicate_flag"].sum()),
                    })
                    continue
                clean, quarantine, audit = adapt_report_table(raw_table, config.project, payload.name)
                canonical_frames.append(clean)
                if not quarantine.empty:
                    quarantined_frames.append(quarantine)
                file_audits.append({"role": "issue_records", **audit})
            except (ValueError, KeyError, pd.errors.ParserError, ET.ParseError) as error:
                errors.append(f"{payload.name}: {type(error).__name__}: {error}")
    if errors:
        LOGGER.error("Source adaptation failed on %d payload(s)", len(errors))
        for error in errors[:15]:
            LOGGER.error("%s", error)
        raise RuntimeError(
            f"Unable to certify complete source ingestion; {len(errors)} file(s) failed. "
            "Correct the schema reader before continuing."
        )
    reports, canonical_audit = consolidate_reports(canonical_frames)
    file_audits.append({"source_file": "CONSOLIDATION", **canonical_audit})
    quarantined = (pd.concat(quarantined_frames, ignore_index=True) if quarantined_frames
                   else pd.DataFrame(columns=REPORT_COLUMNS + ["reason"]))
    official_labels = (pd.concat(official_label_frames, ignore_index=True) if official_label_frames
                       else pd.DataFrame(columns=["record_key", "official_split", "duplicate_flag", "source_file"]))
    if not official_labels.empty:
        if official_labels["record_key"].duplicated().any():
            raise ValueError("Issue appears in multiple official source splits")
        official_labels["matches_canonical_record"] = official_labels["record_key"].isin(reports["record_key"])
        file_audits.append({
            "source_file": "OFFICIAL_LABEL_ALIGNMENT",
            "role": "cross_file_audit",
            "label_rows": len(official_labels),
            "unmatched_labels": int((~official_labels["matches_canonical_record"]).sum()),
            "unlabeled_canonical_reports": int((~reports["record_key"].isin(official_labels["record_key"])).sum()),
        })
    LOGGER.info(
        "Adapted %d issue records and %d official source labels",
        len(reports), len(official_labels),
    )
    return reports, quarantined, file_audits, official_labels


TEST_RESULTS = run_m1_tests()

START_TIME = time.perf_counter()
ARTIFACTS, SOURCE_PROVENANCE = acquire_source(CONFIG)
REPORTS, QUARANTINED, FILE_AUDITS, OFFICIAL_LABELS = load_and_adapt(ARTIFACTS, CONFIG)
if CONFIG.enforce_duplicate_labels and not REPORTS["duplicate_of"].map(parse_duplicate_targets).map(bool).any():
    raise RuntimeError("No source-linked duplicate targets found: cannot certify the positive-label benchmark")
EDGES, DIRECT_EDGES = audit_duplicate_edges(REPORTS)
SPLITS = chronological_splits(REPORTS, CONFIG.train_fraction, CONFIG.validation_fraction)
EDGES = EDGES.drop(columns=["source_split"]).merge(
    SPLITS.rename(columns={"record_key": "source_key", "split": "source_split"}),
    on="source_key", how="left", validate="many_to_one"
)
QUERIES = build_query_registry(REPORTS, EDGES, SPLITS)
DEMO_COHORT = sample_link_preserving_cohort(REPORTS, EDGES, CONFIG.demo_seed_rows)
RUN_SECONDS = time.perf_counter() - START_TIME
LOGGER.info("Canonical reports: %s | links: %s | eligible direct positive links: %s", len(REPORTS), len(EDGES), len(DIRECT_EDGES))

## Data quality gates and exports

A missing target, future target or timestamp tie is retained in the audit—not discarded without explanation. The query registry differentiates verified positive queries from unknown/unconfirmed cases. Release-gate status must be read from the manifest rather than inferred from source-published summary statistics.

In [ ]:
def clean_json_value(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): clean_json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [clean_json_value(item) for item in value]
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        json.dump(clean_json_value(value), stream, indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False)
        stream.write("\n")


def reporting_manifest(config: M1Config) -> dict[str, Any]:
    split_counts = SPLITS["split"].value_counts().to_dict()
    edge_counts = EDGES["status"].value_counts().to_dict()
    eligible_queries = QUERIES.loc[QUERIES["positive_target_count"] > 0]
    by_split = eligible_queries["split"].value_counts().to_dict()
    return {
        "specification": "BT-M1-001", "version": "1.0", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_mode": config.source_mode,
        "real_source_ingested": config.source_mode != "fixture" and len(REPORTS) > 0,
        "source_provenance_verified": config.source_mode == "bughub" and
            SOURCE_PROVENANCE.get("source_kind") == "github_pinned_commit",
        "provenance": SOURCE_PROVENANCE,
        "source_artifacts": [asdict(item) for item in ARTIFACTS],
        "dataset": {
            "report_count": len(REPORTS), "quarantined_report_rows": len(QUARANTINED),
            "total_duplicate_edges": len(EDGES), "edge_status_counts": edge_counts,
            "eligible_direct_positive_queries": len(eligible_queries),
            "eligible_positive_queries_by_split": by_split,
            "chronological_split_counts": split_counts,
            "date_start": REPORTS["created_at"].min(), "date_end": REPORTS["created_at"].max(),
            "sampled_cohort_reports": len(DEMO_COHORT),
            "official_source_label_rows": len(OFFICIAL_LABELS),
            "official_source_label_split_counts": (OFFICIAL_LABELS["official_split"].value_counts().to_dict()
                                                   if not OFFICIAL_LABELS.empty else {}),
            "official_source_label_positive_flags": (int(OFFICIAL_LABELS["duplicate_flag"].sum())
                                                     if not OFFICIAL_LABELS.empty else 0),
            "unmatched_official_label_rows": (int((~OFFICIAL_LABELS["matches_canonical_record"]).sum())
                                              if not OFFICIAL_LABELS.empty else 0),
            "unknown_or_unconfirmed_reports": int((QUERIES["positive_target_count"] == 0).sum()),
        },
        "integrity_gates": {
            "schema_valid": len(REPORTS) > 0,
            "unique_canonical_keys": not REPORTS["record_key"].duplicated().any(),
            "chronological_splits_disjoint": not SPLITS["record_key"].duplicated().any(),
            "all_eligible_positive_links_are_strictly_historical": all(
                REPORTS.set_index("record_key").loc[row.target_key, "created_at"] < row.source_created_at
                for row in DIRECT_EDGES.itertuples(index=False)
            ),
            "verified_positives_present": len(DIRECT_EDGES) > 0,
            "regression_tests_passed": bool(TEST_RESULTS["status"].eq("PASS").all()),
            "real_source_meets_minimum_report_count": (
                True if config.source_mode == "fixture" else len(REPORTS) >= 1_000
            ),
            "real_source_meets_positive_query_count": (
                True if config.source_mode == "fixture" else len(eligible_queries) >= 30
            ),
            "acceptable_quarantine_fraction": (
                len(QUARANTINED) / max(1, len(REPORTS) + len(QUARANTINED)) <= 0.20
            ),
        },
        "benchmark_ready": (
            config.source_mode == "bughub" and len(REPORTS) >= 1_000 and
            len(eligible_queries) >= 30 and
            SOURCE_PROVENANCE.get("source_kind") == "github_pinned_commit"
        ),
        "runtime": {
            "python": sys.version.split()[0], "platform": platform.platform(),
            "pandas": pd.__version__, "numpy": np.__version__, "requests": requests.__version__,
            "ingestion_seconds": round(RUN_SECONDS, 3),
        },
        "limitations": [
            "Duplicate links are retrospective source annotations; first-known labeling dates may be unavailable.",
            "Without creation-time text snapshots, fields may reflect later edits; residual hindsight leakage is disclosed.",
            "Transitive duplicate-group relevance is not inferred during M1.",
            "Missing direct links do not establish verified nonduplicate labels.",
            "BugHub train/test labels are stored as auxiliary source metadata; they are not report bodies or duplicate targets.",
            "The application generates independent chronological train/validation/test partitions.",
            "Source redistribution rights require separate confirmation.",
        ],
    }

MANIFEST = reporting_manifest(CONFIG)
REPORTS.to_csv(NORMALIZED_DIR / "canonical_reports.csv", index=False)
OFFICIAL_LABELS.to_csv(NORMALIZED_DIR / "official_split_labels.csv", index=False)
EDGES.to_csv(NORMALIZED_DIR / "duplicate_edges_audited.csv", index=False)
QUARANTINED.to_csv(NORMALIZED_DIR / "quarantined_records.csv", index=False)
SPLITS.to_csv(NORMALIZED_DIR / "chronological_splits.csv", index=False)
QUERY_EXPORT = QUERIES.copy()
QUERY_EXPORT["positive_target_keys"] = QUERY_EXPORT["positive_target_keys"].map(json.dumps)
QUERY_EXPORT.to_csv(NORMALIZED_DIR / "query_registry.csv", index=False)
DEMO_COHORT.to_csv(NORMALIZED_DIR / "link_preserving_demo_cohort.csv", index=False)
write_json(REPORT_DIR / "source_audit.json", FILE_AUDITS)
write_json(REPORT_DIR / "m1_manifest.json", MANIFEST)
TEST_RESULTS.to_csv(REPORT_DIR / "regression_tests.csv", index=False)

display(pd.DataFrame([MANIFEST["dataset"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame(MANIFEST["integrity_gates"].items(), columns=["gate", "pass"]))
display(REPORTS[["record_key", "created_at", "title", "duplicate_of"]].head(10))
assert all(MANIFEST["integrity_gates"].values()), "M1 release gate failed"

## Dataset inspection

Visual summaries come from the imported records—not the BugHub README headline counts. The cohorts below belong to the current dataset snapshot and must not be mistaken for published benchmark splits.

In [ ]:
def render_m1_summary(reports: pd.DataFrame, edges: pd.DataFrame, splits: pd.DataFrame, destination: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    order = ["train", "validation", "test"]
    counts = splits["split"].value_counts().reindex(order, fill_value=0)
    axes[0].bar(counts.index, counts.to_numpy())
    axes[0].set_title("Chronological cohort counts")
    axes[0].set_ylabel("Bug reports")
    for i, count in enumerate(counts):
        axes[0].text(i, count, str(count), ha="center", va="bottom")
    statuses = edges["status"].value_counts()
    if len(statuses):
        axes[1].barh(statuses.index, statuses.to_numpy())
        axes[1].set_xlabel("Source-annotated duplicate links")
        axes[1].set_title("Duplicate-link audit")
    else:
        axes[1].text(.5, .5, "No linked positives available", ha="center", va="center", transform=axes[1].transAxes)
    fig.tight_layout()
    fig.savefig(destination, dpi=145, bbox_inches="tight")
    plt.show()
    plt.close(fig)

render_m1_summary(REPORTS, EDGES, SPLITS, REPORT_DIR / "m1_summary.png")
print("Validated source mode:", CONFIG.source_mode)
print("Linked-positive evaluation queries:", MANIFEST["dataset"]["eligible_direct_positive_queries"])
print("Completed locally executed regression tests:", len(TEST_RESULTS))
print("Artifacts saved under:", ROOT)

## Lexical retrieval: TF-IDF and BM25

Both retrievers use a fixed vocabulary and corpus statistics derived only from the **training segment** of the selected chronological benchmark population. Eligibility (strict creation-time cutoff and project constraint) is applied **before scoring and top-K ranking**. The configured corpus limit selects the earliest reports in time order, avoiding future-heavy samples; benchmark results from this subset are labeled accordingly.


In [ ]:
from dataclasses import dataclass
from typing import Literal

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

RetrievalMethod = Literal["tfidf", "bm25"]
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9][A-Za-z0-9._/#:+-]*")


def bug_tokens(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.casefold())


@dataclass(frozen=True)
class SearchConfig:
    max_reports: int = int(os.getenv("BUGTWIN_MAX_REPORTS", "12000"))
    max_eval_queries_per_split: int = int(os.getenv("BUGTWIN_EVAL_QUERIES", "80"))
    max_features: int = 60000
    max_query_chars: int = 12000
    max_results: int = 25
    bm25_k1: float = 1.5
    bm25_b: float = 0.75


SEARCH_CONFIG = SearchConfig()
if SEARCH_CONFIG.max_reports < 6 or SEARCH_CONFIG.max_eval_queries_per_split < 1:
    raise ValueError("Retrieval corpus and evaluation limits must be positive")


def select_benchmark_corpus(reports: pd.DataFrame, limit: int) -> pd.DataFrame:
    ordered = reports.sort_values(["created_at", "record_key"], kind="mergesort")
    return ordered.head(limit).reset_index(drop=True).copy()


class LexicalIndex:
    """Frozen-training vocabulary with time-gated candidate search."""

    def __init__(self, reports: pd.DataFrame, train_keys: set[str], config: SearchConfig):
        if reports.empty:
            raise ValueError("Cannot construct a retrieval index with no reports")
        if reports["record_key"].duplicated().any():
            raise ValueError("Cannot index duplicate record identifiers")
        self.reports = reports.reset_index(drop=True).copy()
        self.config = config
        self.keys = self.reports["record_key"].astype(str).to_numpy()
        self.projects = self.reports["project"].astype(str).to_numpy()
        self.dates = pd.to_datetime(self.reports["created_at"], utc=True).dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
        self.key_to_row = {str(key): i for i, key in enumerate(self.keys)}
        self.texts = self.reports["search_text"].fillna("").astype(str).tolist()
        training_positions = np.flatnonzero(self.reports["record_key"].isin(train_keys).to_numpy())
        if not len(training_positions):
            raise ValueError("The vocabulary requires a nonempty historical training partition")
        train_text = [self.texts[i] for i in training_positions]

        self.tfidf = TfidfVectorizer(
            tokenizer=bug_tokens, token_pattern=None, lowercase=False,
            ngram_range=(1, 2), sublinear_tf=True, max_features=config.max_features,
        )
        self.tfidf.fit(train_text)
        self.tfidf_matrix = self.tfidf.transform(self.texts).tocsr()

        self.count = CountVectorizer(
            tokenizer=bug_tokens, token_pattern=None, lowercase=False,
            max_features=config.max_features, dtype=np.int32,
        )
        training_counts = self.count.fit_transform(train_text).tocsr()
        self.count_matrix = self.count.transform(self.texts).tocsr()
        document_frequency = np.asarray((training_counts > 0).sum(axis=0)).ravel()
        n_train = training_counts.shape[0]
        self.idf = np.log1p((n_train - document_frequency + 0.5) / (document_frequency + 0.5))
        self.lengths = np.asarray(self.count_matrix.sum(axis=1)).ravel().astype(np.float64)
        self.average_length = max(float(np.asarray(training_counts.sum(axis=1)).mean()), 1.0)

    def eligible_rows(
        self,
        *,
        before: pd.Timestamp | None = None,
        project: str | None = None,
        exclude_key: str | None = None,
    ) -> np.ndarray:
        allowed = np.ones(len(self.reports), dtype=bool)
        if before is not None:
            cutoff = pd.Timestamp(before)
            if cutoff.tzinfo is None:
                raise ValueError("Historical search requires a timezone-aware cutoff")
            allowed &= self.dates < cutoff.tz_convert("UTC").to_datetime64()
        if project:
            allowed &= self.projects == project
        if exclude_key is not None and exclude_key in self.key_to_row:
            allowed[self.key_to_row[exclude_key]] = False
        return np.flatnonzero(allowed)

    def _bm25_scores(self, query: str, rows: np.ndarray) -> np.ndarray:
        candidates = self.count_matrix[rows]
        lengths = self.lengths[rows]
        norm = self.config.bm25_k1 * (1 - self.config.bm25_b + self.config.bm25_b * lengths / self.average_length)
        scores = np.zeros(len(rows), dtype=np.float64)
        vocabulary = self.count.vocabulary_
        for token in set(bug_tokens(query)):
            position = vocabulary.get(token)
            if position is None:
                continue
            frequencies = candidates[:, position].toarray().ravel().astype(np.float64)
            scores += self.idf[position] * frequencies * (self.config.bm25_k1 + 1) / (frequencies + norm + 1e-12)
        return scores

    def search(
        self, query: str, method: RetrievalMethod = "bm25", top_k: int = 5,
        *, before: pd.Timestamp | None = None, project: str | None = None,
        exclude_key: str | None = None,
    ) -> list[dict[str, Any]]:
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Enter a nonempty bug report")
        if len(query) > self.config.max_query_chars:
            raise ValueError(f"Bug report exceeds {self.config.max_query_chars:,} characters")
        if method not in {"tfidf", "bm25"}:
            raise ValueError(f"Unsupported retrieval method: {method}")
        if not isinstance(top_k, int) or not 1 <= top_k <= self.config.max_results:
            raise ValueError(f"top_k must be in [1, {self.config.max_results}]")
        rows = self.eligible_rows(before=before, project=project, exclude_key=exclude_key)
        if not len(rows):
            return []
        if method == "tfidf":
            query_vector = self.tfidf.transform([query])
            scores = (self.tfidf_matrix[rows] @ query_vector.T).toarray().ravel()
        else:
            scores = self._bm25_scores(query, rows)
        relevant = np.flatnonzero(scores > 1e-12)
        if not len(relevant):
            return []
        selected = relevant[np.lexsort((self.keys[rows[relevant]], -scores[relevant]))][:top_k]
        output: list[dict[str, Any]] = []
        for rank, candidate in enumerate(selected, 1):
            source = self.reports.iloc[int(rows[candidate])]
            output.append({
                "rank": rank,
                "record_key": source["record_key"],
                "title": str(source["title"]),
                "description": str(source["description"]),
                "created_at": pd.Timestamp(source["created_at"]).isoformat(),
                "project": str(source["project"]),
                "component": str(source["component"]),
                "source_url": str(source["source_url"]),
                "score": round(float(scores[candidate]), 6),
                "method": method,
            })
        return output


BENCHMARK_REPORTS = select_benchmark_corpus(REPORTS, SEARCH_CONFIG.max_reports)
BENCHMARK_SPLITS = chronological_splits(
    BENCHMARK_REPORTS, CONFIG.train_fraction, CONFIG.validation_fraction
)
TRAIN_KEYS = set(BENCHMARK_SPLITS.loc[BENCHMARK_SPLITS["split"].eq("train"), "record_key"])
LEXICAL_INDEX = LexicalIndex(BENCHMARK_REPORTS, TRAIN_KEYS, SEARCH_CONFIG)
LOGGER.info(
    "Lexical index built: %d of %d canonical reports; train vocabulary: %d terms",
    len(BENCHMARK_REPORTS), len(REPORTS), len(LEXICAL_INDEX.tfidf.vocabulary_),
)


## Historical retrieval evaluation

The benchmark assesses only queries with **direct, validated earlier-target labels**. Query populations are selected from the configured corpus and distinguished by temporal split. A reduced corpus is a smoke/limited-corpus benchmark, not the full historical BugHub benchmark. No benchmark result is represented as a calibrated duplicate probability.


In [ ]:
def sampled_positive_queries(
    registry: pd.DataFrame, split: str, limit: int,
) -> pd.DataFrame:
    positives = registry.loc[
        registry["split"].eq(split) & registry["positive_target_count"].gt(0)
    ].sort_values(["created_at", "record_key"], kind="mergesort")
    if len(positives) <= limit:
        return positives.reset_index(drop=True)
    positions = np.linspace(0, len(positives) - 1, limit, dtype=int)
    return positives.iloc[positions].reset_index(drop=True)


def retrieval_metrics(results: list[str], positives: set[str], k: int) -> dict[str, float]:
    if not positives:
        raise ValueError("A retrieval query requires at least one verified positive")
    relevant_ranks = [position + 1 for position, key in enumerate(results[:k]) if key in positives]
    hit = float(bool(relevant_ranks))
    reciprocal_rank = 1.0 / relevant_ranks[0] if relevant_ranks else 0.0
    precision_sum = sum(
        sum(key in positives for key in results[:position]) / position
        for position, key in enumerate(results[:k], 1) if key in positives
    )
    return {
        f"Hit@{k}": hit,
        f"MRR@{k}": reciprocal_rank,
        f"Recall@{k}": len(relevant_ranks) / len(positives),
        f"MAP@{k}": precision_sum / min(k, len(positives)),
    }


def evaluate_lexical(
    index: LexicalIndex, registry: pd.DataFrame, methods: tuple[str, ...],
    split: str, limit: int, k: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected = sampled_positive_queries(registry, split, limit)
    rows, examples = [], []
    available = set(index.key_to_row)
    for query in selected.itertuples(index=False):
        positives = set(query.positive_target_keys) & available
        if not positives:
            continue
        source = index.reports.iloc[index.key_to_row[query.record_key]]
        for method in methods:
            found = index.search(
                str(source.search_text), method=method, top_k=k,
                before=pd.Timestamp(source.created_at), project=str(source.project),
                exclude_key=str(source.record_key),
            )
            ranked_keys = [result["record_key"] for result in found]
            metric_values = retrieval_metrics(ranked_keys, positives, k)
            rows.append({
                "method": method, "split": split, "query_key": query.record_key,
                "positive_targets_in_corpus": len(positives),
                "all_direct_targets": len(query.positive_target_keys),
                "eligible_corpus_size": len(index.eligible_rows(before=pd.Timestamp(source.created_at), project=str(source.project))),
                **metric_values,
            })
            examples.append({
                "method": method, "split": split, "query_key": query.record_key,
                "positive_targets": json.dumps(sorted(positives)),
                "retrieved": json.dumps(ranked_keys),
            })
    return pd.DataFrame(rows), pd.DataFrame(examples)


def summarize_results(rows: pd.DataFrame) -> pd.DataFrame:
    if rows.empty:
        return pd.DataFrame(columns=["method", "split", "queries", "Hit@10", "MRR@10", "Recall@10", "MAP@10"])
    metrics = [field for field in rows.columns if field.startswith(("Hit@", "MRR@", "Recall@", "MAP@"))]
    summary = rows.groupby(["method", "split"], as_index=False)[metrics].mean()
    summary.insert(2, "queries", rows.groupby(["method", "split"]).size().to_numpy())
    return summary.round(4)


BENCHMARK_EDGES, _ = audit_duplicate_edges(BENCHMARK_REPORTS)
BENCHMARK_QUERIES = build_query_registry(BENCHMARK_REPORTS, BENCHMARK_EDGES, BENCHMARK_SPLITS)
EVAL_ROWS = []
EVAL_EXAMPLES = []
for phase in ("validation", "test"):
    values, examples = evaluate_lexical(
        LEXICAL_INDEX, BENCHMARK_QUERIES, ("tfidf", "bm25"),
        phase, SEARCH_CONFIG.max_eval_queries_per_split, k=10,
    )
    if not values.empty:
        EVAL_ROWS.append(values)
    if not examples.empty:
        EVAL_EXAMPLES.append(examples)
EVALUATION = pd.concat(EVAL_ROWS, ignore_index=True) if EVAL_ROWS else pd.DataFrame()
EVALUATION_EXAMPLES = pd.concat(EVAL_EXAMPLES, ignore_index=True) if EVAL_EXAMPLES else pd.DataFrame()
EVALUATION_SUMMARY = summarize_results(EVALUATION)
EVALUATION_DIR = ROOT / "reports" / "lexical"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION.to_csv(EVALUATION_DIR / "per_query_metrics.csv", index=False)
EVALUATION_EXAMPLES.to_csv(EVALUATION_DIR / "retrieved_examples.csv", index=False)
EVALUATION_SUMMARY.to_csv(EVALUATION_DIR / "summary.csv", index=False)
write_json(EVALUATION_DIR / "manifest.json", {
    "source_mode": CONFIG.source_mode,
    "source_provenance": SOURCE_PROVENANCE,
    "benchmark_corpus_size": len(BENCHMARK_REPORTS),
    "source_total_reports": len(REPORTS),
    "truncated_corpus": len(BENCHMARK_REPORTS) < len(REPORTS),
    "chronological_vocabulary_training_size": len(TRAIN_KEYS),
    "max_queries_per_split": SEARCH_CONFIG.max_eval_queries_per_split,
    "query_count_by_split": BENCHMARK_QUERIES.groupby("split")["positive_target_count"].apply(lambda s: int(s.gt(0).sum())).to_dict(),
    "evaluated_queries": int(len(EVALUATION) // 2),
    "real_data_benchmark_verified": bool(CONFIG.source_mode == "bughub" and MANIFEST["benchmark_ready"] and not EVALUATION.empty),
    "metric_scope": "historically eligible direct duplicate links; frozen training vocabulary; selected corpus",
})
display(EVALUATION_SUMMARY)
if EVALUATION_SUMMARY.empty:
    LOGGER.warning("No eligible evaluation queries for the chosen cohort; change the corpus cap")


## Retrieval validation

Tests cover temporal gating, deterministic ranking, label handling, frozen training vocabulary, metric definitions, and the real search engine's failure behavior. The synthetic development fixture must never be presented as a measured real-dataset benchmark.


In [ ]:
def test_search_determinism() -> None:
    query = "password login freezes"
    first = LEXICAL_INDEX.search(query, top_k=5)
    second = LEXICAL_INDEX.search(query, top_k=5)
    assert first == second


def test_temporal_eligibility_precedes_ranking() -> None:
    if len(BENCHMARK_REPORTS) < 4:
        return
    cutoff = pd.Timestamp(BENCHMARK_REPORTS.iloc[3].created_at)
    results = LEXICAL_INDEX.search("login password export", before=cutoff, top_k=10)
    assert all(pd.Timestamp(item["created_at"]) < cutoff for item in results)


def test_search_excludes_self() -> None:
    row = BENCHMARK_REPORTS.iloc[-1]
    results = LEXICAL_INDEX.search(str(row.search_text), exclude_key=str(row.record_key))
    assert all(item["record_key"] != row.record_key for item in results)


def test_search_ranking_scores() -> None:
    for method in ("tfidf", "bm25"):
        values = LEXICAL_INDEX.search("error login", method=method, top_k=10)
        assert all(item["score"] > 0 for item in values)
        assert [item["score"] for item in values] == sorted([item["score"] for item in values], reverse=True)


def test_benchmark_direct_labels_only() -> None:
    positives = BENCHMARK_QUERIES.loc[BENCHMARK_QUERIES["positive_target_count"].gt(0)]
    assert all(all(target in LEXICAL_INDEX.key_to_row for target in targets) for targets in positives.positive_target_keys)


def test_historical_label_eligibility() -> None:
    lookup = BENCHMARK_REPORTS.set_index("record_key")["created_at"]
    for query in BENCHMARK_QUERIES.loc[BENCHMARK_QUERIES["positive_target_count"].gt(0)].itertuples(index=False):
        assert all(lookup[target] < query.created_at for target in query.positive_target_keys)


def test_metric_known_fixture() -> None:
    metrics = retrieval_metrics(["X", "A", "B", "C"], {"A", "C"}, 4)
    assert metrics["Hit@4"] == 1.0
    assert metrics["MRR@4"] == 0.5
    assert metrics["Recall@4"] == 1.0
    assert abs(metrics["MAP@4"] - 0.5) < 1e-10


def test_metric_rejects_unknown_labels() -> None:
    try:
        retrieval_metrics(["A"], set(), 10)
    except ValueError:
        return
    raise AssertionError("Unknown-label query was scored as a negative")


def test_invalid_inputs() -> None:
    for query in ("", " ", None):
        try:
            LEXICAL_INDEX.search(query)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid query accepted")
    for count in (-1, 0, SEARCH_CONFIG.max_results + 1):
        try:
            LEXICAL_INDEX.search("error", top_k=count)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid top_k accepted")


def test_project_filter() -> None:
    project = str(BENCHMARK_REPORTS.iloc[0].project)
    results = LEXICAL_INDEX.search("login error", project=project)
    assert all(item["project"] == project for item in results)


def test_no_future_vocabulary() -> None:
    training_text = BENCHMARK_REPORTS.loc[BENCHMARK_REPORTS["record_key"].isin(TRAIN_KEYS), "search_text"]
    tokens = {term for text in training_text for term in bug_tokens(str(text))}
    withheld = next((term for term in bug_tokens(str(BENCHMARK_REPORTS.iloc[-1].search_text)) if term not in tokens), None)
    if withheld is not None:
        assert withheld not in LEXICAL_INDEX.count.vocabulary_


def test_search_schema() -> None:
    values = LEXICAL_INDEX.search("bug error", top_k=3)
    required = {"rank", "record_key", "title", "created_at", "project", "score", "method"}
    assert all(required.issubset(item) for item in values)


SEARCH_TESTS = [
    test_search_determinism,
    test_temporal_eligibility_precedes_ranking,
    test_search_excludes_self,
    test_search_ranking_scores,
    test_benchmark_direct_labels_only,
    test_historical_label_eligibility,
    test_metric_known_fixture,
    test_metric_rejects_unknown_labels,
    test_invalid_inputs,
    test_project_filter,
    test_no_future_vocabulary,
    test_search_schema,
]
SEARCH_TEST_RESULTS = []
for check in SEARCH_TESTS:
    try:
        check()
        SEARCH_TEST_RESULTS.append({"test": check.__name__, "status": "PASS", "details": ""})
    except Exception as error:
        SEARCH_TEST_RESULTS.append({"test": check.__name__, "status": "FAIL", "details": f"{type(error).__name__}: {error}"})
SEARCH_TEST_FRAME = pd.DataFrame(SEARCH_TEST_RESULTS)
SEARCH_TEST_FRAME.to_csv(EVALUATION_DIR / "retrieval_tests.csv", index=False)
display(SEARCH_TEST_FRAME)
assert SEARCH_TEST_FRAME["status"].eq("PASS").all(), "Lexical retrieval regression suite failed"


## Semantic retrieval and hybrid ranking

M3 extends the existing M1/M2 pipeline without modifying the canonical report schema. A pretrained MiniLM encoder produces normalized document and query embeddings; semantic search uses FAISS `IndexFlatIP` for unrestricted live search and **exact, prefiltered cosine ranking** for historical and project-restricted queries. That filtered path is necessary to enforce the time boundary before top-K selection. BM25 and dense rankings are combined with Reciprocal Rank Fusion (RRF). An MS MARCO cross-encoder can be enabled separately for reranking, but its raw outputs are relevance scores rather than duplicate probabilities.

For CI, `BUGTWIN_M1_SOURCE=fixture` selects a deterministic token-hashing encoder. **It is a test double, not a semantic language model.** Real-data execution uses `sentence-transformers/all-MiniLM-L6-v2`. If the neural encoder cannot be loaded, lexical retrieval remains operational and the error is disclosed.


In [ ]:
from typing import Protocol


@dataclass(frozen=True)
class M3Config:
    encoder_id: str = os.getenv("BUGTWIN_ENCODER_ID", "sentence-transformers/all-MiniLM-L6-v2")
    reranker_id: str = os.getenv("BUGTWIN_RERANKER_ID", "cross-encoder/ms-marco-MiniLM-L6-v2")
    encoder_mode: str = os.getenv("BUGTWIN_ENCODER_MODE", "auto").strip().lower()
    batch_size: int = int(os.getenv("BUGTWIN_EMBED_BATCH_SIZE", "32"))
    candidate_pool: int = int(os.getenv("BUGTWIN_CANDIDATE_POOL", "20"))
    rerank_pool: int = int(os.getenv("BUGTWIN_RERANK_POOL", "12"))
    rrf_k: int = 60
    enable_reranker: bool = os.getenv("BUGTWIN_ENABLE_RERANKER", "0") == "1"


M3_CONFIG = M3Config()
if M3_CONFIG.encoder_mode not in {"auto", "model", "fixture", "off"}:
    raise ValueError("BUGTWIN_ENCODER_MODE must be auto, model, fixture, or off")
if M3_CONFIG.batch_size < 1 or not 1 <= M3_CONFIG.candidate_pool <= SEARCH_CONFIG.max_results:
    raise ValueError("Invalid M3 embedding batch size or candidate pool")
if not 1 <= M3_CONFIG.rerank_pool <= M3_CONFIG.candidate_pool:
    raise ValueError("Invalid M3 rerank pool")
if M3_CONFIG.rrf_k < 1:
    raise ValueError("RRF smoothing must be positive")


class TextEncoder(Protocol):
    identifier: str

    def encode(self, texts: list[str], *, batch_size: int) -> np.ndarray:
        ...


class HashFixtureEncoder:
    """Deterministic test double; never a substitute for a pretrained model."""

    identifier = "fixture:hashed-technical-tokens"

    def __init__(self, dimensions: int = 128) -> None:
        self.dimensions = dimensions

    def encode(self, texts: list[str], *, batch_size: int) -> np.ndarray:
        result = np.zeros((len(texts), self.dimensions), dtype=np.float32)
        for i, text in enumerate(texts):
            tokens = bug_tokens(text)
            for token in tokens:
                digest = hashlib.sha256(token.encode("utf-8")).digest()
                position = int.from_bytes(digest[:4], "little") % self.dimensions
                result[i, position] += 1.0
        return result


class MiniLMEncoder:
    def __init__(self, model_id: str) -> None:
        from sentence_transformers import SentenceTransformer
        self.identifier = model_id
        self.model = SentenceTransformer(model_id)

    def encode(self, texts: list[str], *, batch_size: int) -> np.ndarray:
        values = self.model.encode(
            texts, batch_size=batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        )
        return np.asarray(values, dtype=np.float32)


def normalized_embeddings(values: np.ndarray, expected_rows: int) -> np.ndarray:
    matrix = np.asarray(values, dtype=np.float32)
    if matrix.ndim != 2 or matrix.shape[0] != expected_rows or matrix.shape[1] == 0:
        raise ValueError(f"Invalid embedding matrix shape: {matrix.shape}")
    if not np.isfinite(matrix).all():
        raise ValueError("Nonfinite embedding values")
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    matrix = matrix / np.maximum(norms, 1e-12)
    return np.ascontiguousarray(matrix, dtype=np.float32)


def fingerprint_documents(reports: pd.DataFrame, encoder_id: str) -> str:
    digest = hashlib.sha256(encoder_id.encode("utf-8"))
    for record in reports[["record_key", "search_text"]].itertuples(index=False):
        digest.update(json.dumps([str(record.record_key), str(record.search_text)], ensure_ascii=False).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


class SemanticIndex:
    """Exact cosine retrieval with eligibility applied before candidate ranking."""

    def __init__(
        self, lexical: LexicalIndex, encoder: TextEncoder, config: M3Config,
        cache_dir: Path | None = None, use_faiss: bool = True,
    ) -> None:
        self.lexical = lexical
        self.encoder = encoder
        self.config = config
        self.report_count = len(lexical.reports)
        self.fingerprint = fingerprint_documents(lexical.reports, encoder.identifier)
        cache_path = cache_dir / f"{self.fingerprint}.npz" if cache_dir is not None else None
        vectors = None
        if cache_path is not None and cache_path.exists():
            with np.load(cache_path, allow_pickle=False) as payload:
                if str(payload["fingerprint"].item()) == self.fingerprint:
                    vectors = normalized_embeddings(payload["vectors"], self.report_count)
                    LOGGER.info("Loaded semantic embeddings from %s", cache_path)
        if vectors is None:
            vectors = normalized_embeddings(
                encoder.encode(lexical.texts, batch_size=config.batch_size),
                self.report_count,
            )
            if cache_path is not None:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = cache_path.with_name(cache_path.stem + ".partial.npz")
                np.savez_compressed(temporary, vectors=vectors, fingerprint=self.fingerprint)
                temporary.replace(cache_path)
        self.vectors = vectors
        self.faiss_index = None
        self.backend = "exact_numpy"
        if use_faiss:
            try:
                import faiss
                index = faiss.IndexFlatIP(vectors.shape[1])
                index.add(vectors)
                if index.ntotal != self.report_count:
                    raise RuntimeError("FAISS index size mismatch")
                self.faiss_index = index
                self.backend = "faiss_flat_ip"
            except ImportError:
                LOGGER.warning("FAISS unavailable; exact NumPy cosine retrieval remains enabled")

    def search(
        self, query: str, top_k: int = 5, *, before: pd.Timestamp | None = None,
        project: str | None = None, exclude_key: str | None = None,
    ) -> list[dict[str, Any]]:
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Enter a nonempty bug report")
        if len(query) > SEARCH_CONFIG.max_query_chars:
            raise ValueError(f"Bug report exceeds {SEARCH_CONFIG.max_query_chars:,} characters")
        if not isinstance(top_k, int) or not 1 <= top_k <= SEARCH_CONFIG.max_results:
            raise ValueError(f"top_k must be in [1, {SEARCH_CONFIG.max_results}]")
        eligible = self.lexical.eligible_rows(before=before, project=project, exclude_key=exclude_key)
        if len(eligible) == 0:
            return []
        query_vector = normalized_embeddings(self.encoder.encode([query], batch_size=1), 1)
        if float(np.linalg.norm(query_vector[0])) < 1e-8:
            return []
        count = min(top_k, len(eligible))
        if self.faiss_index is not None and len(eligible) == self.report_count:
            scores, indices = self.faiss_index.search(query_vector, count)
            selected_rows = indices[0]
            selected_scores = scores[0]
            valid = selected_rows >= 0
            selected_rows = selected_rows[valid]
            selected_scores = selected_scores[valid]
        else:
            # Rank only the eligible subset: filtering retrieved top-K afterwards is invalid.
            similarities = self.vectors[eligible] @ query_vector[0]
            order = np.lexsort((self.lexical.keys[eligible], -similarities))[:count]
            selected_rows = eligible[order]
            selected_scores = similarities[order]
        order = np.lexsort((self.lexical.keys[selected_rows], -selected_scores))
        output = []
        for rank, position in enumerate(order, 1):
            row = self.lexical.reports.iloc[int(selected_rows[position])]
            output.append({
                "rank": rank, "record_key": str(row.record_key),
                "title": str(row.title), "description": str(row.description),
                "created_at": pd.Timestamp(row.created_at).isoformat(),
                "project": str(row.project), "component": str(row.component),
                "source_url": str(row.source_url),
                "score": round(float(selected_scores[position]), 6),
                "method": "semantic", "backend": self.backend if len(eligible) == self.report_count else "prefiltered_exact_cosine",
            })
        return output


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]], smoothing: int = 60,
) -> list[dict[str, Any]]:
    if smoothing < 1:
        raise ValueError("smoothing must be positive")
    candidates: dict[str, dict[str, Any]] = {}
    for ranked in ranked_lists:
        seen: set[str] = set()
        for rank, item in enumerate(ranked, 1):
            key = str(item["record_key"])
            if key in seen:
                continue
            seen.add(key)
            if key not in candidates:
                candidates[key] = {**item, "rrf_score": 0.0, "signals": {}}
            candidates[key]["rrf_score"] += 1.0 / (smoothing + rank)
            candidates[key]["signals"][str(item["method"])] = float(item["score"])
    merged = sorted(candidates.values(), key=lambda item: (-item["rrf_score"], item["record_key"]))
    for rank, item in enumerate(merged, 1):
        item["rank"] = rank
        item["method"] = "hybrid"
        item["score"] = round(item["rrf_score"], 8)
    return merged


class CrossEncoderReranker:
    def __init__(self, model_id: str) -> None:
        from sentence_transformers import CrossEncoder
        self.identifier = model_id
        self.model = CrossEncoder(model_id)

    def score(self, query: str, candidates: list[dict[str, Any]]) -> np.ndarray:
        pairs = [(query, f"{item['title']}\n{item['description']}"[:4000]) for item in candidates]
        return np.asarray(self.model.predict(pairs, batch_size=16, show_progress_bar=False), dtype=np.float32).reshape(-1)


class HybridIndex:
    def __init__(
        self, lexical: LexicalIndex, semantic: SemanticIndex | None,
        config: M3Config, reranker: Any = None,
    ) -> None:
        self.lexical = lexical
        self.semantic = semantic
        self.config = config
        self.reranker = reranker

    @property
    def methods(self) -> list[str]:
        methods = ["bm25", "tfidf"]
        if self.semantic is not None:
            methods += ["semantic", "hybrid"]
            if self.reranker is not None:
                methods.append("hybrid_rerank")
        return methods

    def search(
        self, query: str, method: str = "hybrid", top_k: int = 5,
        *, before: pd.Timestamp | None = None,
        project: str | None = None, exclude_key: str | None = None,
    ) -> list[dict[str, Any]]:
        if method not in self.methods:
            raise ValueError(f"Method {method!r} is unavailable; available: {', '.join(self.methods)}")
        if not isinstance(top_k, int) or not 1 <= top_k <= SEARCH_CONFIG.max_results:
            raise ValueError(f"top_k must be in [1, {SEARCH_CONFIG.max_results}]")
        restrictions = {"before": before, "project": project, "exclude_key": exclude_key}
        if method in {"bm25", "tfidf"}:
            return self.lexical.search(query, method=method, top_k=top_k, **restrictions)
        if method == "semantic":
            return self.semantic.search(query, top_k=top_k, **restrictions)
        pool = max(top_k, self.config.candidate_pool)
        lexical = self.lexical.search(query, method="bm25", top_k=pool, **restrictions)
        dense = self.semantic.search(query, top_k=pool, **restrictions)
        ranked = reciprocal_rank_fusion([lexical, dense], self.config.rrf_k)
        if method == "hybrid_rerank" and ranked:
            head = ranked[:self.config.rerank_pool]
            scores = np.asarray(self.reranker.score(query, head), dtype=np.float32)
            if scores.shape != (len(head),) or not np.isfinite(scores).all():
                raise ValueError("Reranker returned invalid scores")
            for item, score in zip(head, scores):
                item["reranker_score"] = float(score)
            head.sort(key=lambda item: (-item["reranker_score"], item["record_key"]))
            ranked = head + ranked[self.config.rerank_pool:]
        output = []
        for rank, item in enumerate(ranked[:top_k], 1):
            record = {**item, "rank": rank, "method": method}
            if method == "hybrid_rerank" and "reranker_score" in record:
                record["score"] = round(record["reranker_score"], 6)
            output.append(record)
        return output


mode = M3_CONFIG.encoder_mode
if mode == "auto":
    mode = "fixture" if CONFIG.source_mode == "fixture" else "model"
if mode == "fixture" and CONFIG.source_mode != "fixture":
    raise ValueError("Synthetic embeddings may only be used with fixture source data")
EMBEDDING_KIND = "disabled"
SEMANTIC_ERROR = ""
SEMANTIC_INDEX: SemanticIndex | None = None
RERANKER = None
if mode != "off":
    try:
        encoder = HashFixtureEncoder() if mode == "fixture" else MiniLMEncoder(M3_CONFIG.encoder_id)
        SEMANTIC_INDEX = SemanticIndex(
            LEXICAL_INDEX, encoder, M3_CONFIG, cache_dir=ROOT / "cache" / "embeddings",
        )
        EMBEDDING_KIND = "synthetic_fixture" if mode == "fixture" else "pretrained_sentence_transformer"
    except (ImportError, OSError, RuntimeError, ValueError) as error:
        SEMANTIC_ERROR = f"{type(error).__name__}: {error}"
        LOGGER.error("Semantic retrieval unavailable: %s. Lexical search remains operational.", SEMANTIC_ERROR)
if SEMANTIC_INDEX is not None and M3_CONFIG.enable_reranker and mode == "model":
    try:
        RERANKER = CrossEncoderReranker(M3_CONFIG.reranker_id)
    except (ImportError, OSError, RuntimeError, ValueError) as error:
        LOGGER.warning("Cross-encoder unavailable; hybrid search remains operational: %s", error)
HYBRID_INDEX = HybridIndex(LEXICAL_INDEX, SEMANTIC_INDEX, M3_CONFIG, RERANKER)
LOGGER.info("M3 encoder=%s | backend=%s | methods=%s", EMBEDDING_KIND,
            SEMANTIC_INDEX.backend if SEMANTIC_INDEX else "none", HYBRID_INDEX.methods)


## Semantic and hybrid evaluation

The comparison reuses the same eligible, historical positive queries and corpus from M2. Only reports created before each query may be retrieved. Per-query metrics, latency, model configuration, and source lineage are exported under `bugtwin_runtime/reports/m3/`. Fixture metrics are explicitly marked as **pipeline tests, not NLP performance results**. A pretrained-model failure does not silently substitute the synthetic encoder for real data.


In [ ]:
def evaluate_m3(
    index: HybridIndex, registry: pd.DataFrame, split: str, limit: int,
    k: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    queries = sampled_positive_queries(registry, split, limit)
    per_query, candidates = [], []
    available = set(index.lexical.key_to_row)
    for query in queries.itertuples(index=False):
        targets = set(query.positive_target_keys) & available
        if not targets:
            continue
        source = index.lexical.reports.iloc[index.lexical.key_to_row[query.record_key]]
        for method in index.methods:
            start = time.perf_counter()
            results = index.search(
                str(source.search_text), method=method, top_k=k,
                before=pd.Timestamp(source.created_at), project=str(source.project),
                exclude_key=str(source.record_key),
            )
            duration = (time.perf_counter() - start) * 1000
            ranked = [item["record_key"] for item in results]
            per_query.append({
                "split": split, "method": method, "query_key": query.record_key,
                "direct_targets": len(targets), "latency_ms": round(duration, 3),
                **retrieval_metrics(ranked, targets, k),
            })
            candidates.append({
                "split": split, "method": method, "query_key": query.record_key,
                "positive_targets": json.dumps(sorted(targets)),
                "retrieved": json.dumps(ranked),
            })
    return pd.DataFrame(per_query), pd.DataFrame(candidates)


M3_DIR = ROOT / "reports" / "m3"
M3_DIR.mkdir(parents=True, exist_ok=True)
M3_ROWS, M3_EXAMPLES = [], []
for phase in ("validation", "test"):
    query_rows, retrieved = evaluate_m3(
        HYBRID_INDEX, BENCHMARK_QUERIES, phase,
        SEARCH_CONFIG.max_eval_queries_per_split, 10,
    )
    if not query_rows.empty:
        M3_ROWS.append(query_rows)
        M3_EXAMPLES.append(retrieved)
M3_EVALUATION = pd.concat(M3_ROWS, ignore_index=True) if M3_ROWS else pd.DataFrame()
M3_RETRIEVED = pd.concat(M3_EXAMPLES, ignore_index=True) if M3_EXAMPLES else pd.DataFrame()
M3_SUMMARY = summarize_results(M3_EVALUATION)
if not M3_SUMMARY.empty:
    latency = M3_EVALUATION.groupby(["method", "split"], as_index=False)["latency_ms"].median()
    M3_SUMMARY = M3_SUMMARY.merge(latency.rename(columns={"latency_ms": "median_latency_ms"}),
                                  on=["method", "split"], validate="one_to_one")
M3_EVALUATION.to_csv(M3_DIR / "per_query_metrics.csv", index=False)
M3_RETRIEVED.to_csv(M3_DIR / "retrieved_candidates.csv", index=False)
M3_SUMMARY.to_csv(M3_DIR / "summary.csv", index=False)
M3_MANIFEST = {
    "milestone": "M3", "source_mode": CONFIG.source_mode,
    "embedding_kind": EMBEDDING_KIND,
    "encoder_id": SEMANTIC_INDEX.encoder.identifier if SEMANTIC_INDEX else None,
    "semantic_error": SEMANTIC_ERROR or None,
    "faiss_backend": SEMANTIC_INDEX.backend if SEMANTIC_INDEX else None,
    "reranker_id": getattr(RERANKER, "identifier", None),
    "methods_evaluated": HYBRID_INDEX.methods,
    "corpus_fingerprint": SEMANTIC_INDEX.fingerprint if SEMANTIC_INDEX else None,
    "corpus_size": len(BENCHMARK_REPORTS),
    "corpus_truncated": len(BENCHMARK_REPORTS) < len(REPORTS),
    "dataset_provenance": SOURCE_PROVENANCE,
    "rrf_k": M3_CONFIG.rrf_k, "candidate_pool": M3_CONFIG.candidate_pool,
    "evaluated_rows": len(M3_EVALUATION),
    "real_neural_benchmark_verified": bool(
        CONFIG.source_mode == "bughub" and MANIFEST["benchmark_ready"]
        and EMBEDDING_KIND == "pretrained_sentence_transformer"
        and not M3_EVALUATION.empty
    ),
    "interpretation": "Fixture-mode scores are integration tests, not model-quality measurements.",
    "limitations": [
        "Direct confirmed historical duplicate links only; unlabeled bugs are not verified negatives.",
        "Candidate text may include retrospective edits when the source lacks text snapshots.",
        "FAISS is used for unrestricted search; restricted historical searches use exact prefiltered cosine ranking.",
        "Reranker outputs are relevance scores, not calibrated duplicate probabilities.",
    ],
}
write_json(M3_DIR / "manifest.json", M3_MANIFEST)
display(M3_SUMMARY)


## M3 regression and integration tests

Tests execute against deterministic fixture embeddings in CI and the real pretrained encoder in Colab. The fixture encoder checks contracts, filtering, ranking, and UI integration without being presented as an NLP benchmark. Missing FAISS is handled by exact cosine search; when FAISS is installed, its full-corpus output is compared with the exact reference.


In [ ]:
class ControlledReranker:
    identifier = "fixture:controlled-reranker"

    def score(self, query: str, candidates: list[dict[str, Any]]) -> np.ndarray:
        return np.asarray([float(i) for i in range(len(candidates))], dtype=np.float32)


def test_m3_encoder_contract() -> None:
    model = HashFixtureEncoder()
    one = normalized_embeddings(model.encode(["A.java HTTP 401", "login failure"], batch_size=2), 2)
    two = normalized_embeddings(model.encode(["A.java HTTP 401", "login failure"], batch_size=1), 2)
    np.testing.assert_allclose(one, two)
    np.testing.assert_allclose(np.linalg.norm(one, axis=1), np.ones(2), rtol=1e-5)


def test_m3_rejects_invalid_embedding_shapes() -> None:
    for matrix in [np.zeros(3), np.zeros((2, 0)), np.full((2, 4), np.nan)]:
        try:
            normalized_embeddings(matrix, 2)
        except ValueError:
            continue
        raise AssertionError("Invalid embedding matrix accepted")


def test_m3_corpus_fingerprint() -> None:
    base = fingerprint_documents(BENCHMARK_REPORTS, "encoder-a")
    assert base == fingerprint_documents(BENCHMARK_REPORTS.copy(), "encoder-a")
    assert base != fingerprint_documents(BENCHMARK_REPORTS, "encoder-b")


def test_m3_temporal_restriction() -> None:
    index = SemanticIndex(LEXICAL_INDEX, HashFixtureEncoder(), M3_CONFIG, use_faiss=False)
    cutoff = pd.Timestamp(BENCHMARK_REPORTS.iloc[-2].created_at)
    for result in index.search("login error", before=cutoff, top_k=10):
        assert pd.Timestamp(result["created_at"]) < cutoff


def test_m3_candidate_filter_precedes_rank() -> None:
    index = SemanticIndex(LEXICAL_INDEX, HashFixtureEncoder(), M3_CONFIG, use_faiss=False)
    row = BENCHMARK_REPORTS.iloc[-1]
    results = index.search(str(row.search_text), top_k=5, before=pd.Timestamp(row.created_at),
                           exclude_key=str(row.record_key), project=str(row.project))
    assert all(result["record_key"] != row.record_key for result in results)
    assert all(result["project"] == row.project for result in results)
    assert all(pd.Timestamp(result["created_at"]) < row.created_at for result in results)


def test_m3_prefiltered_cosine_matches_bruteforce() -> None:
    index = SemanticIndex(LEXICAL_INDEX, HashFixtureEncoder(), M3_CONFIG, use_faiss=False)
    q = "authentication password frozen login"
    cutoff = pd.Timestamp(BENCHMARK_REPORTS.iloc[-1].created_at)
    eligible = LEXICAL_INDEX.eligible_rows(before=cutoff)
    v = normalized_embeddings(index.encoder.encode([q], batch_size=1), 1)[0]
    similarity = index.vectors[eligible] @ v
    expected = [str(LEXICAL_INDEX.keys[i]) for i in eligible[np.lexsort((LEXICAL_INDEX.keys[eligible], -similarity))[:5]]]
    found = [item["record_key"] for item in index.search(q, before=cutoff, top_k=5)]
    assert found == expected


def test_m3_search_is_deterministic() -> None:
    if SEMANTIC_INDEX is not None:
        assert SEMANTIC_INDEX.search("export error", top_k=5) == SEMANTIC_INDEX.search("export error", top_k=5)
        assert HYBRID_INDEX.search("export error", method="hybrid", top_k=5) == HYBRID_INDEX.search("export error", method="hybrid", top_k=5)


def test_m3_rrf_ranks_consistently() -> None:
    a = [{"record_key": "a", "method": "bm25", "score": 2.0}, {"record_key": "b", "method": "bm25", "score": 1.0}]
    b = [{"record_key": "b", "method": "semantic", "score": 0.8}, {"record_key": "c", "method": "semantic", "score": 0.3}]
    merged = reciprocal_rank_fusion([a, b])
    assert [item["record_key"] for item in merged] == ["b", "a", "c"]
    assert merged[0]["signals"].keys() == {"bm25", "semantic"}


def test_m3_reranking_changes_priority() -> None:
    test_index = SemanticIndex(LEXICAL_INDEX, HashFixtureEncoder(), M3_CONFIG, use_faiss=False)
    hybrid = HybridIndex(LEXICAL_INDEX, test_index, M3_CONFIG, ControlledReranker())
    query = "login password"
    basic = hybrid.search(query, method="hybrid", top_k=5)
    updated = hybrid.search(query, method="hybrid_rerank", top_k=5)
    assert {item["record_key"] for item in updated}.issubset(
        {item["record_key"] for item in hybrid.search(query, method="hybrid", top_k=M3_CONFIG.rerank_pool)}
    )
    if len(basic) > 1:
        assert updated[0]["record_key"] != basic[0]["record_key"]
    assert all(item["method"] == "hybrid_rerank" for item in updated)


def test_m3_invalid_options() -> None:
    for method in ("fake", "hybrid_rerank" if RERANKER is None else "invalid"):
        try:
            HYBRID_INDEX.search("login", method=method)
        except ValueError:
            continue
        raise AssertionError("Unavailable search method accepted")
    if SEMANTIC_INDEX is not None:
        for query in ("", None):
            try:
                SEMANTIC_INDEX.search(query)
            except ValueError:
                continue
            raise AssertionError("Invalid semantic query accepted")
    for count in (0, -1, SEARCH_CONFIG.max_results + 1, 3.5):
        try:
            HYBRID_INDEX.search("login", method="bm25", top_k=count)
        except ValueError:
            continue
        raise AssertionError("Invalid hybrid top_k accepted")


def test_m3_interface_output_contract() -> None:
    assert "bm25" in HYBRID_INDEX.methods and "tfidf" in HYBRID_INDEX.methods
    if SEMANTIC_INDEX is not None:
        output = HYBRID_INDEX.search("login failure", method="semantic", top_k=3)
        assert len(output) <= 3
        assert all({"record_key", "score", "method", "title"}.issubset(item) for item in output)


def test_m3_faiss_equivalence_when_available() -> None:
    if SEMANTIC_INDEX is None or SEMANTIC_INDEX.faiss_index is None:
        return
    query = "login password network timeout"
    faiss_results = SEMANTIC_INDEX.search(query, top_k=5)
    reference = SemanticIndex(LEXICAL_INDEX, SEMANTIC_INDEX.encoder, M3_CONFIG,
                              cache_dir=ROOT / "cache" / "embeddings", use_faiss=False)
    numpy_results = reference.search(query, top_k=5)
    np.testing.assert_allclose(
        [item["score"] for item in faiss_results],
        [item["score"] for item in numpy_results], rtol=1e-5, atol=1e-5,
    )
    assert all(item["record_key"] in LEXICAL_INDEX.key_to_row for item in faiss_results)


M3_TESTS = [
    test_m3_encoder_contract, test_m3_rejects_invalid_embedding_shapes,
    test_m3_corpus_fingerprint, test_m3_temporal_restriction,
    test_m3_candidate_filter_precedes_rank, test_m3_prefiltered_cosine_matches_bruteforce,
    test_m3_search_is_deterministic, test_m3_rrf_ranks_consistently,
    test_m3_reranking_changes_priority, test_m3_invalid_options,
    test_m3_interface_output_contract, test_m3_faiss_equivalence_when_available,
]
M3_TEST_RESULTS = []
for check in M3_TESTS:
    try:
        check()
        M3_TEST_RESULTS.append({"test": check.__name__, "status": "PASS", "details": ""})
    except Exception as error:
        M3_TEST_RESULTS.append({"test": check.__name__, "status": "FAIL", "details": f"{type(error).__name__}: {error}"})
M3_TEST_FRAME = pd.DataFrame(M3_TEST_RESULTS)
M3_TEST_FRAME.to_csv(M3_DIR / "regression_tests.csv", index=False)
display(M3_TEST_FRAME)
assert M3_TEST_FRAME["status"].eq("PASS").all(), "M3 regression suite failed"


## Interactive duplicate search

The interface exposes the retrievers available in this runtime. It retains lexical search if pretrained-model loading fails, includes source metadata in results, and marks fixture-mode search as a pipeline demonstration. Scores are ranking signals, not probabilities; neither a high score nor a missing match proves that reports are duplicates or distinct.


In [ ]:
import gradio as gr


SEARCH_COLUMNS = ["Rank", "Report", "Title", "Score", "Method", "Created", "Component", "Source URL", "Excerpt"]


def search_for_interface(
    title: str, description: str, method: str, top_k: float,
) -> tuple[pd.DataFrame, str]:
    query = "\n".join(item.strip() for item in (title or "", description or "") if item and item.strip())
    if not query:
        return pd.DataFrame(columns=SEARCH_COLUMNS), "Enter a bug title or description."
    try:
        results = HYBRID_INDEX.search(query, method=method, top_k=int(top_k))
    except (ValueError, RuntimeError) as error:
        return pd.DataFrame(columns=SEARCH_COLUMNS), str(error)
    data = pd.DataFrame([{
        "Rank": item["rank"], "Report": item["record_key"],
        "Title": item["title"], "Score": item["score"],
        "Method": item["method"], "Created": item["created_at"][:10],
        "Component": item["component"], "Source URL": item["source_url"],
        "Excerpt": item["description"][:250],
    } for item in results], columns=SEARCH_COLUMNS)
    message = (
        f"Found {len(results)} candidate reports using **{method}**. "
        "Review source issues before deciding whether they are duplicates."
        if results else
        "No candidates found under this search configuration. This does not establish that the bug is new."
    )
    if EMBEDDING_KIND == "synthetic_fixture":
        message += " **Fixture-mode embeddings are synthetic test signals, not NLP predictions.**"
    if SEMANTIC_ERROR:
        message += " **Semantic retrieval is unavailable; lexical search remains active.**"
    return data, message


with gr.Blocks(title="BugTwin — Duplicate Bug Report Search") as BUGTWIN_APP:
    gr.Markdown("# BugTwin\n**Duplicate bug-report retrieval** · lexical, semantic, and hybrid search")
    gr.Markdown(
        f"Indexed **{len(BENCHMARK_REPORTS):,}** reports · Source: `{CONFIG.source_mode}` · "
        f"Encoder: `{EMBEDDING_KIND}` · "
        "Scores are ranking signals, not duplicate probabilities."
    )
    with gr.Tab("Find potential duplicates"):
        input_title = gr.Textbox(label="Bug title", placeholder="Example: Login freezes after password reset")
        input_description = gr.Textbox(label="Detailed description", lines=5,
                                       placeholder="Symptoms, reproduction steps, environment, and error messages")
        with gr.Row():
            input_method = gr.Dropdown(HYBRID_INDEX.methods, label="Retrieval method",
                                       value="hybrid" if "hybrid" in HYBRID_INDEX.methods else "bm25")
            input_k = gr.Slider(minimum=1, maximum=SEARCH_CONFIG.max_results, step=1, value=5, label="Results")
        search_button = gr.Button("Find related bug reports", variant="primary")
        search_message = gr.Markdown("Enter a bug report to search the indexed corpus.")
        search_table = gr.Dataframe(label="Candidates", interactive=False, wrap=True)
        search_button.click(
            search_for_interface,
            inputs=[input_title, input_description, input_method, input_k],
            outputs=[search_table, search_message],
        )
    with gr.Tab("Model evaluation"):
        gr.Markdown("**Chronological retrieval evaluation.** Every query searches only earlier reports with verified direct duplicate links.")
        if CONFIG.source_mode == "fixture":
            gr.Markdown("*Synthetic fixture results validate code paths, not real NLP model quality.*")
        if len(BENCHMARK_REPORTS) < len(REPORTS):
            gr.Markdown("*The current benchmark uses a capped corpus, not all available reports.*")
        gr.Dataframe(value=M3_SUMMARY, label="Method comparison (Hit@10, MRR@10, Recall@10, MAP@10, latency)", interactive=False)
        gr.Markdown("Results and manifests are exported under `bugtwin_runtime/reports/m3/`.")
    with gr.Tab("Dataset information"):
        overview = pd.DataFrame([
            {"Property": "Source mode", "Value": CONFIG.source_mode},
            {"Property": "Canonical reports", "Value": len(REPORTS)},
            {"Property": "Indexed reports", "Value": len(BENCHMARK_REPORTS)},
            {"Property": "Embedding kind", "Value": EMBEDDING_KIND},
            {"Property": "Vector backend", "Value": SEMANTIC_INDEX.backend if SEMANTIC_INDEX else "unavailable"},
            {"Property": "Available retrieval methods", "Value": ", ".join(HYBRID_INDEX.methods)},
            {"Property": "Real-data benchmark ready", "Value": M3_MANIFEST["real_neural_benchmark_verified"]},
        ])
        gr.Dataframe(value=overview, label="Corpus and model details", interactive=False)
        gr.Markdown("Source: https://github.com/logpai/bughub · Downloaded report data is not included in this repository.")

if os.getenv("BUGTWIN_LAUNCH_UI", "1" if "google.colab" in sys.modules else "0") == "1":
    share_link = os.getenv(
        "BUGTWIN_SHARE_UI",
        "1" if "google.colab" in sys.modules and CONFIG.source_mode in {"bughub", "fixture"} else "0",
    ) == "1"
    BUGTWIN_APP.launch(share=share_link, prevent_thread_lock=True, show_error=False)
else:
    LOGGER.info("Gradio interface constructed; automatic launch disabled")


## Runtime output

Data tables are written under `bugtwin_runtime/data/normalized/`; experiment and test results are written under `bugtwin_runtime/reports/`. The repository must never commit datasets, runtime caches, pretrained models, or interface uploads. The standalone notebook remains the sole application source file.
